# Notebook 3 (Publication Edition): Strain-Specific Capability Analysis

This notebook is the publication-ready integration of the final plotting workflow.
It keeps the original computational logic unchanged and standardizes documentation in English.

Main modules:
1. Baseline FBA and flux export
2. Formate-light scanning with heatmaps and linear regression
3. dFBA cell-factory simulation (lycopene route)
4. Pie chart generation and pie-chart workflow flowchart


## Inputs / Outputs

### Input model
- `../Models/purple_bacteriav_DSM123.json` (final curated model)

### Output directory
- `publication_outputs/`

### Core generated files
- `flux_full.csv` / `flux_exchange_nonzero.csv`
- `Heatmap_Growth_Rate.svg`
- `Heatmap_Carbon_Efficiency.svg`
- `Analysis_Linear_Regression.svg`
- `dFBA_CellFactory_Lycopene.svg`
- `dFBA_CellFactory_Summary.csv`
- `Carbon_Partition_Pie_Average.svg`


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import cobra
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib import rcParams
from matplotlib.patches import FancyBboxPatch
from IPython.display import display

# --- Global style (publication-oriented) ---
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['mathtext.fontset'] = 'custom'
rcParams['mathtext.rm'] = 'Arial'
rcParams['mathtext.it'] = 'Arial:italic'
rcParams['mathtext.bf'] = 'Arial:bold'
rcParams['axes.unicode_minus'] = False

MODEL_JSON = Path('../Models/purple_bacteriav_DSM123.json')
OUTPUT_DIR = Path('publication_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

assert MODEL_JSON.exists(), f'Model file not found: {MODEL_JSON}'
print(f'Model: {MODEL_JSON.resolve()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')


Model: D:\Project\ai_project\DSM123\old\../Models/purple_bacteriav_DSM123.json
Output dir: D:\Project\ai_project\DSM123\old\publication_outputs


In [2]:

# ---------- Shared model setup ----------

def apply_base_medium(model, oxygen_lb=-0.1, formate_bounds=(-1000.0, 1000.0), photon_lb=-1000.0):
    """Apply base physicochemical constraints used across analyses."""
    for ex in model.exchanges:
        ex.lower_bound = 0.0

    essential_ions = [
        'EX_cobalt2_e', 'EX_zn2_e', 'EX_so4_e', 'EX_ca2_e', 'EX_mn2_e',
        'EX_mg2_e', 'EX_cu2_e', 'EX_k_e', 'EX_fe3_e', 'EX_mobd_e',
        'EX_na1_e', 'EX_cl_e', 'EX_bo3_e', 'EX_pi_e', 'EX_h2o_e', 'EX_h_e', 'EX_nh4_e'
    ]
    for ion in essential_ions:
        if ion in model.reactions:
            model.reactions.get_by_id(ion).lower_bound = -1000.0

    if 'EX_co2_e' in model.reactions:
        model.reactions.get_by_id('EX_co2_e').bounds = (-1000.0, 1000.0)
    if 'EX_o2_e' in model.reactions:
        model.reactions.get_by_id('EX_o2_e').bounds = (oxygen_lb, 1000.0)
    if 'EX_for_e' in model.reactions:
        model.reactions.get_by_id('EX_for_e').bounds = formate_bounds
    if 'NTRSA' in model.reactions:
        model.reactions.get_by_id('NTRSA').bounds = (0.0, 0.0)
    if 'EX_photon_purple_e' in model.reactions:
        model.reactions.get_by_id('EX_photon_purple_e').bounds = (photon_lb, 0.0)

    # Prevent shortcut fluxes used in previous notebooks
    if 'POR5' in model.reactions:
        model.reactions.get_by_id('POR5').lower_bound = 0.0
    if 'ME2' in model.reactions:
        model.reactions.get_by_id('ME2').lower_bound = 0.0

    return model


def ensure_lycopene_demand(model):
    """Add demand reaction DM_lycop_c if lycopene metabolite exists."""
    try:
        lycop = model.metabolites.get_by_id('lycop_c')
        if 'DM_lycop_c' not in model.reactions:
            model.add_boundary(lycop, type='demand')
    except KeyError:
        print('Warning: lycop_c not found in model, DM_lycop_c not added.')
    return model


## 1) Baseline FBA and Flux Export

In [3]:

model = cobra.io.load_json_model(str(MODEL_JSON))
model = apply_base_medium(model, oxygen_lb=-0.1, formate_bounds=(-20.0, 1000.0), photon_lb=-1000.0)
solution = model.optimize()

flux_df = solution.to_frame().reset_index().rename(columns={'index': 'reaction_id'})
flux_df.to_csv(OUTPUT_DIR / 'flux_full.csv', index=False, encoding='utf-8-sig')

exchange_df = flux_df[
    flux_df['reaction_id'].str.startswith('EX_') & (flux_df['fluxes'].abs() > 1e-9)
].sort_values('fluxes', ascending=False)
exchange_df.to_csv(OUTPUT_DIR / 'flux_exchange_nonzero.csv', index=False, encoding='utf-8-sig')

print(f'Objective value (growth): {solution.objective_value:.6f}')
display(exchange_df.head(20))


Objective value (growth): 0.447596


,reaction_id,fluxes,reduced_costs
430,EX_h2o_e,1.448874e+01,0.000000e+00
432,EX_co2_e,1.054401e+01,0.000000e+00
478,EX_pi_e,1.632266e-01,0.000000e+00
451,EX_mobd_e,-9.796930e-09,0.000000e+00
447,EX_cu2_e,-1.350811e-08,0.000000e+00
2084,EX_bo3_e,-1.632741e-08,0.000000e+00
453,EX_na1_e,-5.326394e-08,-1.016319e-09
442,EX_mn2_e,-7.879776e-08,0.000000e+00
435,EX_zn2_e,-3.293191e-07,0.000000e+00
448,EX_k_e,-6.924312e-07,0.000000e+00


## 2) Formate-Light Scan (Heatmaps + Linear Regression)


In [4]:

def run_formate_light_scan(model_path=MODEL_JSON, grid_size=50):
    model = cobra.io.load_json_model(str(model_path))
    model = apply_base_medium(model, oxygen_lb=-0.1, formate_bounds=(-1000.0, 1000.0), photon_lb=-1000.0)

    formate_range = np.linspace(1.0, 12.0, grid_size)
    light_range = np.linspace(1.0, 65.0, grid_size)

    res_biomass = np.zeros((grid_size, grid_size))
    res_efficiency = np.zeros((grid_size, grid_size))

    for i, for_val in enumerate(formate_range):
        with model as m:
            if 'EX_for_e' in m.reactions:
                m.reactions.get_by_id('EX_for_e').lower_bound = -float(for_val)
            for j, light_val in enumerate(light_range):
                if 'EX_photon_purple_e' in m.reactions:
                    m.reactions.get_by_id('EX_photon_purple_e').lower_bound = -float(light_val)

                obj = m.slim_optimize()
                if obj and obj > 1e-9:
                    sol = m.optimize()
                    res_biomass[i, j] = sol.objective_value
                    v_fdh = float(sol.fluxes['FDH']) if 'FDH' in sol.fluxes else 0.0
                    v_rbpc = float(sol.fluxes['RBPC']) if 'RBPC' in sol.fluxes else 0.0
                    res_efficiency[i, j] = v_rbpc / v_fdh if abs(v_fdh) > 1e-9 else 0.0

    # Save raw matrices
    np.save(OUTPUT_DIR / 'res_biomass.npy', res_biomass)
    np.save(OUTPUT_DIR / 'res_efficiency.npy', res_efficiency)
    np.save(OUTPUT_DIR / 'ranges.npy', {'formate': formate_range, 'light': light_range})

    pd.DataFrame(res_biomass, index=formate_range, columns=light_range).to_csv(
        OUTPUT_DIR / 'res_biomass_matrix.csv', encoding='utf-8-sig'
    )
    pd.DataFrame(res_efficiency, index=formate_range, columns=light_range).to_csv(
        OUTPUT_DIR / 'res_efficiency_matrix.csv', encoding='utf-8-sig'
    )

    return formate_range, light_range, res_biomass, res_efficiency


def plot_scan_results(formate_range, light_range, res_biomass, res_efficiency):
    grid_size = len(formate_range)
    tick_idx = np.linspace(0, grid_size - 1, 6).astype(int)
    x_labels = [int(light_range[k]) for k in tick_idx]
    y_labels = [int(formate_range[k]) for k in tick_idx]
    mask = (res_biomass < 1e-3)

    # Figure A: growth heatmap
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    sns.heatmap(res_biomass, ax=ax1, cmap='RdBu_r', mask=mask,
                cbar_kws={'label': r'Growth Rate ($\mathrm{h}^{-1}$)'})
    ax1.set_xlabel(r'Light Uptake ($\mathrm{mmol\ gDCW}^{-1}\ \mathrm{h}^{-1}$)', fontsize=13, fontweight='bold')
    ax1.set_ylabel(r'Formate Uptake ($\mathrm{mmol\ gDCW}^{-1}\ \mathrm{h}^{-1}$)', fontsize=13, fontweight='bold')
    ax1.set_xticks(tick_idx + 0.5); ax1.set_yticks(tick_idx + 0.5)
    ax1.set_xticklabels(x_labels, rotation=0); ax1.set_yticklabels(y_labels, rotation=0)
    ax1.invert_yaxis()
    fig1.tight_layout()
    fig1.savefig(OUTPUT_DIR / 'Heatmap_Growth_Rate.svg', format='svg', bbox_inches='tight')
    plt.close(fig1)

    # Figure B: carbon-efficiency heatmap
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    sns.heatmap(res_efficiency, ax=ax2, cmap='RdBu_r', mask=mask,
                cbar_kws={'label': 'RBPC / FDH Ratio'})
    ax2.set_xlabel(r'Light Uptake ($\mathrm{mmol\ gDCW}^{-1}\ \mathrm{h}^{-1}$)', fontsize=13, fontweight='bold')
    ax2.set_ylabel(r'Formate Uptake ($\mathrm{mmol\ gDCW}^{-1}\ \mathrm{h}^{-1}$)', fontsize=13, fontweight='bold')
    ax2.set_xticks(tick_idx + 0.5); ax2.set_yticks(tick_idx + 0.5)
    ax2.set_xticklabels(x_labels, rotation=0); ax2.set_yticklabels(y_labels, rotation=0)
    ax2.invert_yaxis()
    fig2.tight_layout()
    fig2.savefig(OUTPUT_DIR / 'Heatmap_Carbon_Efficiency.svg', format='svg', bbox_inches='tight')
    plt.close(fig2)

    # Figure C: linear regression of stoichiometric inflection points
    inflection_lights, valid_formates = [], []
    for i in range(grid_size):
        row = res_biomass[i, :]
        max_v = np.max(row)
        if max_v < 1e-2:
            continue
        idx = np.where(row >= 0.99 * max_v)[0][0]
        inflection_lights.append(light_range[idx])
        valid_formates.append(formate_range[i])

    if len(inflection_lights) >= 2:
        slope, intercept, r_value, p_value, _ = stats.linregress(inflection_lights, valid_formates)
        r2 = r_value ** 2

        fig3, ax3 = plt.subplots(figsize=(10, 8))
        ax3.scatter(inflection_lights, valid_formates, color='black', alpha=0.65, s=45, label='Inflection points')
        fit_x = np.array([min(inflection_lights), max(inflection_lights)])
        fit_y = slope * fit_x + intercept
        ax3.plot(fit_x, fit_y, '--', color='#d62728', linewidth=2.5,
                 label=f'Fit: y={slope:.4f}x+({intercept:.3f}), $R^2$={r2:.4f}')
        ax3.set_xlabel(r'Light Uptake ($\mathrm{mmol\ gDCW}^{-1}\ \mathrm{h}^{-1}$)', fontsize=13, fontweight='bold')
        ax3.set_ylabel(r'Formate Uptake ($\mathrm{mmol\ gDCW}^{-1}\ \mathrm{h}^{-1}$)', fontsize=13, fontweight='bold')
        ax3.grid(True, linestyle=':', alpha=0.6)
        ax3.legend(frameon=True)
        fig3.tight_layout()
        fig3.savefig(OUTPUT_DIR / 'Analysis_Linear_Regression.svg', format='svg', bbox_inches='tight')
        plt.close(fig3)

        pd.DataFrame({
            'light_inflection': inflection_lights,
            'formate_at_inflection': valid_formates
        }).to_csv(OUTPUT_DIR / 'Stoichiometric_Inflection_Points.csv', index=False, encoding='utf-8-sig')

        print(f'Linear regression: Formate = {slope:.5f} * Light + ({intercept:.5f}), R^2={r2:.5f}, p={p_value:.2e}')
    else:
        print('Not enough valid points for linear regression.')


In [5]:

formate_range, light_range, res_biomass, res_efficiency = run_formate_light_scan(MODEL_JSON, grid_size=50)
plot_scan_results(formate_range, light_range, res_biomass, res_efficiency)
print('Scan figures and tables saved to publication_outputs/.')


Linear regression: Formate = 0.19611 * Light + (-0.09957), R^2=0.99946, p=4.19e-80
Scan figures and tables saved to publication_outputs/.


## 3) dFBA Cell Factory Simulation (Lycopene Route)

### dFBA Hyperparameters (Editable)
Edit this dictionary before running to test different dFBA settings.


In [6]:
DFBA_PARAMS = {
    'initial_formates': (5, 10, 15, 20),
    't_max': 72,
    'dt': 0.5,
    'vmax_for': 20.0,
    'km_for': 1.0,
    'x0': 0.05,
    'total_light': 100.0,
    'max_light': 60.0,
    'qp_lycop': 0.02,
    'c_biomass': 40.0,
    'c_lycop': 40.0,
}
display(pd.DataFrame([DFBA_PARAMS]))


,initial_formates,t_max,dt,vmax_for,km_for,x0,total_light,max_light,qp_lycop,c_biomass,c_lycop
0,"(5, 10, 15, 20)",72,0.5,20.0,1.0,0.05,100.0,60.0,0.02,40.0,40.0


In [7]:

def run_dfba_cell_factory(
    model_path=MODEL_JSON,
    initial_formates=(5, 10, 15, 20),
    t_max=72,
    dt=0.5,
    vmax_for=20.0,
    km_for=1.0,
    x0=0.05,
    total_light=100.0,
    max_light=60.0,
    qp_lycop=0.02,
    c_biomass=40.0,
    c_lycop=40.0,
):
    model = cobra.io.load_json_model(str(model_path))
    model = apply_base_medium(model, oxygen_lb=-1.0, formate_bounds=(-1000.0, 1000.0), photon_lb=-1000.0)
    model = ensure_lycopene_demand(model)
    model.objective = 'BIOMASS__1'

    time_steps = np.arange(0, t_max + dt, dt)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    results = {}

    for for0 in initial_formates:
        X, For, CO2, Lycop = x0, float(for0), 0.0, 0.0
        hist_x, hist_for, hist_co2, hist_lycop = [X], [For], [CO2], [Lycop]

        for _ in time_steps[:-1]:
            uptake = vmax_for * (For / (km_for + For)) if For > 0 else 0.0
            light = min(total_light / max(X, 1e-9), max_light)

            model.reactions.get_by_id('EX_for_e').lower_bound = -uptake
            model.reactions.get_by_id('EX_photon_purple_e').lower_bound = -light
            if 'DM_lycop_c' in model.reactions:
                model.reactions.get_by_id('DM_lycop_c').lower_bound = qp_lycop

            obj = model.slim_optimize()
            if obj and obj > 1e-9:
                sol = model.optimize()
                mu = float(sol.objective_value)
                flux_for = float(sol.fluxes.get('EX_for_e', 0.0))
                flux_co2 = float(sol.fluxes.get('EX_co2_e', 0.0))
                flux_lycop = float(sol.fluxes.get('DM_lycop_c', 0.0))
            else:
                mu, flux_for, flux_co2, flux_lycop = 0.0, 0.0, 0.0, 0.0

            X = X + mu * X * dt
            For = max(0.0, For + flux_for * X * dt)
            CO2 = CO2 + flux_co2 * X * dt
            Lycop = Lycop + flux_lycop * X * dt

            hist_x.append(X)
            hist_for.append(For)
            hist_co2.append(CO2)
            hist_lycop.append(Lycop)

        results[for0] = {
            'time': time_steps,
            'X': hist_x,
            'For': hist_for,
            'CO2': hist_co2,
            'Lycop': hist_lycop,
        }

    # 4-panel figure
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    ax_a = axes[0, 0]
    for idx, f0 in enumerate(initial_formates):
        ax_a.plot(results[f0]['time'], results[f0]['X'], color=colors[idx], lw=2.2, label=f'{f0} mM')
    ax_a.set_title('a', loc='left', fontsize=16, fontweight='bold')
    ax_a.set_xlabel('Fermentation time (h)')
    ax_a.set_ylabel(r'Biomass concentration (gDW $\mathrm{L}^{-1}$)')
    ax_a.grid(True, alpha=0.3)
    ax_a.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=4, frameon=False)

    ax_b = axes[0, 1]
    for idx, f0 in enumerate(initial_formates):
        ax_b.plot(results[f0]['time'], results[f0]['CO2'], color=colors[idx], lw=2.2, label=f'{f0} mM')
    ax_b.set_title('b', loc='left', fontsize=16, fontweight='bold')
    ax_b.set_xlabel('Fermentation time (h)')
    ax_b.set_ylabel(r'$\mathrm{CO}_2$ released (mM)')
    ax_b.grid(True, alpha=0.3)
    ax_b.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=4, frameon=False)

    ax_c1 = axes[1, 0]
    ax_c2 = ax_c1.twinx()
    x_pos = np.arange(len(initial_formates))
    width = 0.35

    final_biomass = [results[f]['X'][-1] for f in initial_formates]
    co2_yield = [results[f]['CO2'][-1] / (f - results[f]['For'][-1] + 1e-6) for f in initial_formates]

    ax_c1.bar(x_pos - width / 2, final_biomass, width, color='#4c72b0', label='Final biomass')
    ax_c2.bar(x_pos + width / 2, co2_yield, width, color='#dd8452', label=r'$\mathrm{CO}_2$ yield')
    ax_c1.set_title('c', loc='left', fontsize=16, fontweight='bold')
    ax_c1.set_xlabel('Formate feed concentration (mM)')
    ax_c1.set_ylabel(r'Final biomass (gDW $\mathrm{L}^{-1}$)', color='#4c72b0')
    ax_c2.set_ylabel(r'$\mathrm{CO}_2$ yield (mol $\mathrm{mol}^{-1}$ formate)', color='#dd8452')
    ax_c1.set_xticks(x_pos)
    ax_c1.set_xticklabels(initial_formates)
    h1, l1 = ax_c1.get_legend_handles_labels()
    h2, l2 = ax_c2.get_legend_handles_labels()
    ax_c1.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)

    # Carbon partition stacked bar (biomass / CO2 / lycopene)
    ax_d = axes[1, 1]
    C_biomass, C_co2, C_lycop = [], [], []
    for f in initial_formates:
        c_bio = (results[f]['X'][-1] - x0) * c_biomass
        c_co2 = results[f]['CO2'][-1]
        c_lyc = results[f]['Lycop'][-1] * c_lycop
        total_c = c_bio + c_co2 + c_lyc
        if total_c > 0:
            C_biomass.append(c_bio / total_c * 100)
            C_co2.append(c_co2 / total_c * 100)
            C_lycop.append(c_lyc / total_c * 100)
        else:
            C_biomass.append(0.0); C_co2.append(0.0); C_lycop.append(0.0)

    ax_d.bar(x_pos, C_biomass, width=0.6, color='#4c72b0', label='Biomass')
    ax_d.bar(x_pos, C_co2, bottom=C_biomass, width=0.6, color='#9e9ac8', label=r'$\mathrm{CO}_2$')
    ax_d.bar(x_pos, C_lycop, bottom=np.array(C_biomass)+np.array(C_co2), width=0.6, color='#d62728', label='Lycopene')
    ax_d.set_title('d', loc='left', fontsize=16, fontweight='bold')
    ax_d.set_xlabel('Formate feed concentration (mM)')
    ax_d.set_ylabel('Formate carbon partition (%)')
    ax_d.set_xticks(x_pos)
    ax_d.set_xticklabels(initial_formates)
    ax_d.set_ylim(0, 100)
    ax_d.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'dFBA_CellFactory_Lycopene.svg', format='svg', dpi=300, bbox_inches='tight')
    plt.close(fig)

    # Save summary table
    rows = []
    for idx, f in enumerate(initial_formates):
        rows.append({
            'Formate_mM': f,
            'Final_Biomass_gDW_L': results[f]['X'][-1],
            'Final_CO2_mM': results[f]['CO2'][-1],
            'Final_Lycopene_mM': results[f]['Lycop'][-1],
            'Carbon_to_Biomass_pct': C_biomass[idx],
            'Carbon_to_CO2_pct': C_co2[idx],
            'Carbon_to_Lycopene_pct': C_lycop[idx],
        })
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(OUTPUT_DIR / 'dFBA_CellFactory_Summary.csv', index=False, encoding='utf-8-sig')

    return results, summary_df


dfba_results, dfba_summary = run_dfba_cell_factory(model_path=MODEL_JSON, **DFBA_PARAMS)
display(dfba_summary)


,Formate_mM,Final_Biomass_gDW_L,Final_CO2_mM,Final_Lycopene_mM,Carbon_to_Biomass_pct,Carbon_to_CO2_pct,Carbon_to_Lycopene_pct
0,5,0.125239,2.896777,0.008344,48.229215,46.421891,5.348893
1,10,0.201709,5.991666,0.014469,48.013844,47.406993,4.579163
2,15,0.276545,9.034245,0.019978,47.958269,47.812436,4.229295
3,20,0.356440,12.275602,0.026841,47.868442,47.938752,4.192805


## 4) Pie Chart


In [8]:

def plot_carbon_partition_pie(summary_df, output_dir=OUTPUT_DIR):
    avg = summary_df[
        ['Carbon_to_Biomass_pct', 'Carbon_to_CO2_pct', 'Carbon_to_Lycopene_pct']
    ].mean()

    labels = ['Biomass', 'CO2', 'Lycopene']
    values = [avg['Carbon_to_Biomass_pct'], avg['Carbon_to_CO2_pct'], avg['Carbon_to_Lycopene_pct']]
    colors = ['#4c72b0', '#9e9ac8', '#d62728']

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=90,
           colors=colors, wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
           textprops={'fontsize': 12})
    ax.set_title('Average Carbon Partition (dFBA Cell Factory)', fontsize=14, fontweight='bold')
    fig.tight_layout()
    fig.savefig(output_dir / 'Carbon_Partition_Pie_Average.svg', format='svg', bbox_inches='tight')
    plt.close(fig)


plot_carbon_partition_pie(dfba_summary, OUTPUT_DIR)
print('Pie chart exported.')


Pie chart exported.


## 5) Output Manifest

In [9]:

out_files = sorted([p.name for p in OUTPUT_DIR.glob('*')])
print(f'Total output files: {len(out_files)}')
for f in out_files:
    print('-', f)


Total output files: 14
- Analysis_Linear_Regression.svg
- Carbon_Partition_Pie_Average.svg
- Heatmap_Carbon_Efficiency.svg
- Heatmap_Growth_Rate.svg
- Stoichiometric_Inflection_Points.csv
- dFBA_CellFactory_Lycopene.svg
- dFBA_CellFactory_Summary.csv
- flux_exchange_nonzero.csv
- flux_full.csv
- ranges.npy
- res_biomass.npy
- res_biomass_matrix.csv
- res_efficiency.npy
- res_efficiency_matrix.csv


## 6) Michaelis-Menten Enzyme-Constrained dFBA for the CBB and TCA Core Loop

This section applies enzyme-kinetic constraints during every dFBA time step. For each CBB/TCA reaction, the effective flux bound is recalculated with a Michaelis-Menten expression, `v = E * kcat * S / (Km + S)`. Multi-substrate reactions use the most limiting saturation term. Intracellular CBB/TCA substrate concentrations are exposed as editable assumptions because the current dFBA state tracks biomass, formate, and CO2 rather than full metabolite pools.



In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import cobra
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display

mpl.rcParams['svg.fonttype'] = 'none'

MM_DFBA_OUTPUT_DIR = Path('.')
MM_DFBA_OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_JSON = Path('../Models/purple_bacteriav_DSM123.json')


def apply_base_conditions(model):
    """Apply the shared medium and reaction constraints used for the dFBA runs."""
    for ex in model.exchanges:
        ex.lower_bound = 0.0
    essential_ions = [
        'EX_cobalt2_e', 'EX_zn2_e', 'EX_so4_e', 'EX_ca2_e', 'EX_mn2_e',
        'EX_mg2_e', 'EX_cu2_e', 'EX_k_e', 'EX_fe3_e', 'EX_mobd_e',
        'EX_na1_e', 'EX_cl_e', 'EX_bo3_e', 'EX_pi_e', 'EX_h2o_e', 'EX_h_e', 'EX_nh4_e'
    ]
    for ion in essential_ions:
        if ion in model.reactions:
            model.reactions.get_by_id(ion).lower_bound = -1000.0
    if 'EX_co2_e' in model.reactions:
        model.reactions.get_by_id('EX_co2_e').bounds = (-1000.0, 1000.0)
    if 'EX_o2_e' in model.reactions:
        model.reactions.get_by_id('EX_o2_e').bounds = (-1.0, 1000.0)
    if 'NTRSA' in model.reactions:
        model.reactions.get_by_id('NTRSA').bounds = (0, 0.0)
    if 'BIOMASS__1' in model.reactions:
        model.reactions.BIOMASS__1.upper_bound = 0.023645071
    if 'EX_photon_purple_e' in model.reactions:
        model.reactions.get_by_id('EX_photon_purple_e').bounds = (-1000.0, 0.0)
    if 'POR5' in model.reactions:
        model.reactions.get_by_id('POR5').lower_bound = 0.0
    if 'ME2' in model.reactions:
        model.reactions.get_by_id('ME2').lower_bound = 0.0
    return model


def _extract_locus_tags(gpr):
    return sorted(set(re.findall(r'ACXYSJ_\d+', str(gpr))))


def _load_rpal_homolog_map():
    mapping_path = Path('DSM123_blast_gene_rename_output_rpal65/rpal_to_DSM123_mapping_threshold65.csv')
    if not mapping_path.exists():
        return {}
    mapping = pd.read_csv(mapping_path)
    result = {}
    for _, row in mapping.iterrows():
        target = str(row.get('primary_target', ''))
        if target.startswith('ACXYSJ_'):
            result.setdefault(target, []).append(str(row.get('rpal_gene', '')))
    return {key: ';'.join(sorted(set(val for val in vals if val and val != 'nan'))) for key, vals in result.items()}


def _load_dsm_product_map():
    product_path = Path('DSM123_blast_gene_rename_output_rpal_pidmax/DSM123_annotation_reference_with_sequence.csv')
    if not product_path.exists():
        return {}
    ann = pd.read_csv(product_path)
    return ann.set_index('dsm_locus_tag')[['gene', 'product', 'protein_id', 'aa_length']].to_dict('index')


def substrate(metabolite_id, km_mM, default_mM, source='assumed intracellular pool'):
    return {'metabolite_id': metabolite_id, 'km_mM': km_mM, 'default_mM': default_mM, 'source': source}


# Kinetic values are placed in an explicit table so UniProt-reviewed values can be replaced directly.
# The dFBA solver uses these values through a Michaelis-Menten equation at every time step.
KINETIC_ASSUMPTIONS = {
    'RBPC': {
        'pathway': 'CBB', 'ec_number': '4.1.1.39', 'enzyme': 'Ribulose-bisphosphate carboxylase',
        'kcat_s_inv': 3.0, 'enzyme_umol_per_gdw': 0.01,
        'substrates': [substrate('co2_c', 0.02, 0.01, 'dynamic CO2 proxy'), substrate('rb15bp_c', 0.02, 0.10)],
        'uniprot_accession_or_query': 'query: cbbL/rbcL/rbcS Rubisco Rhodopseudomonas palustris',
        'kinetic_source': 'UniProtKB/BRENDA-compatible Michaelis-Menten parameter slot; replace with accession-specific reviewed values when available',
    },
    'RBCh': {
        'pathway': 'CBB', 'ec_number': '4.1.1.39', 'enzyme': 'Rubisco oxygenase side activity',
        'kcat_s_inv': 1.0, 'enzyme_umol_per_gdw': 0.01,
        'substrates': [substrate('o2_c', 0.02, 0.25, 'assumed dissolved oxygen proxy'), substrate('rb15bp_c', 0.02, 0.10)],
        'uniprot_accession_or_query': 'query: Rubisco oxygenase Rhodopseudomonas palustris',
        'kinetic_source': 'UniProtKB/BRENDA-compatible Michaelis-Menten parameter slot; replace with accession-specific reviewed values when available',
    },
    'RBPCcx': {
        'pathway': 'CBB', 'ec_number': '4.1.1.39', 'enzyme': 'Rubisco carboxylase/oxygenase mixed reaction',
        'kcat_s_inv': 3.0, 'enzyme_umol_per_gdw': 0.01,
        'substrates': [substrate('co2_c', 0.02, 0.01, 'dynamic CO2 proxy'), substrate('rb15bp_c', 0.02, 0.10)],
        'uniprot_accession_or_query': 'query: Rubisco mixed carboxylase oxygenase Rhodopseudomonas palustris',
        'kinetic_source': 'UniProtKB/BRENDA-compatible Michaelis-Menten parameter slot; replace with accession-specific reviewed values when available',
    },
    'FDH': {
        'pathway': 'Formate oxidation', 'ec_number': '1.17.1.9', 'enzyme': 'Formate dehydrogenase',
        'kcat_s_inv': 4.87, 'enzyme_umol_per_gdw': 0.01,
        'substrates': [substrate('for_c', 0.2378, 10.0, 'dynamic formate proxy')],
        'uniprot_accession_or_query': 'Q6NBU4',
        'kinetic_source': 'RpFDH parameters from this study: kcat=4.87 s^-1, Km(formate)=237.8 uM (0.2378 mM), kcat/Km=0.0205 s^-1 uM^-1; CGA009 homolog UniProt Q6NBU4 is used only as a reference accession',
        'rnaseq_locus_tags_override': ['PGIDNB_23465', 'PGIDNB_23470', 'PGIDNB_23475', 'PGIDNB_23480', 'PGIDNB_23485'],
        'dsm_locus_tags_override': ['ACXYSJ_23600', 'ACXYSJ_23605', 'ACXYSJ_23610', 'ACXYSJ_23615', 'ACXYSJ_23620'],
    },
    'RPE': {'pathway': 'CBB', 'ec_number': '5.1.3.1', 'enzyme': 'Ribulose-phosphate 3-epimerase', 'kcat_s_inv': 200.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('ru5p__D_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: rpe Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'FBA': {'pathway': 'CBB', 'ec_number': '4.1.2.13', 'enzyme': 'Fructose-bisphosphate aldolase', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('fdp_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: fba Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'FBA3': {'pathway': 'CBB', 'ec_number': '4.1.2.13', 'enzyme': 'Sedoheptulose-bisphosphate aldolase activity', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('s17bp_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: sedoheptulose bisphosphate aldolase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'PGK': {'pathway': 'CBB', 'ec_number': '2.7.2.3', 'enzyme': 'Phosphoglycerate kinase', 'kcat_s_inv': 250.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('3pg_c', 0.10, 0.10), substrate('atp_c', 0.10, 2.00)], 'uniprot_accession_or_query': 'query: pgk Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'GAPD': {'pathway': 'CBB', 'ec_number': '1.2.1.12', 'enzyme': 'Glyceraldehyde-3-phosphate dehydrogenase', 'kcat_s_inv': 100.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('g3p_c', 0.10, 0.10), substrate('nad_c', 0.10, 1.00)], 'uniprot_accession_or_query': 'query: gapA/gapD Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'TKT1': {'pathway': 'CBB', 'ec_number': '2.2.1.1', 'enzyme': 'Transketolase', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('r5p_c', 0.10, 0.10), substrate('xu5p__D_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: transketolase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'TKT2': {'pathway': 'CBB', 'ec_number': '2.2.1.1', 'enzyme': 'Transketolase', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('e4p_c', 0.10, 0.10), substrate('xu5p__D_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: transketolase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'TALA': {'pathway': 'CBB', 'ec_number': '2.2.1.2', 'enzyme': 'Transaldolase', 'kcat_s_inv': 50.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('g3p_c', 0.10, 0.10), substrate('s7p_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: transaldolase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'SBP': {'pathway': 'CBB', 'ec_number': '3.1.3.37', 'enzyme': 'Sedoheptulose-bisphosphatase', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('s17bp_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: sedoheptulose bisphosphatase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'FBP': {'pathway': 'CBB', 'ec_number': '3.1.3.11', 'enzyme': 'Fructose-bisphosphatase', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('fdp_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: fructose bisphosphatase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'CS': {'pathway': 'TCA', 'ec_number': '2.3.3.1', 'enzyme': 'Citrate synthase', 'kcat_s_inv': 60.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('accoa_c', 0.10, 0.20), substrate('oaa_c', 0.05, 0.05)], 'uniprot_accession_or_query': 'query: gltA citrate synthase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'ACONT': {'pathway': 'TCA', 'ec_number': '4.2.1.3', 'enzyme': 'Aconitate hydratase', 'kcat_s_inv': 50.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('cit_c', 0.10, 0.20)], 'uniprot_accession_or_query': 'query: aconitate hydratase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'ACONTa': {'pathway': 'TCA', 'ec_number': '4.2.1.3', 'enzyme': 'Aconitase half-reaction A', 'kcat_s_inv': 50.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('cit_c', 0.10, 0.20)], 'uniprot_accession_or_query': 'query: aconitase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'ACONTb': {'pathway': 'TCA', 'ec_number': '4.2.1.3', 'enzyme': 'Aconitase half-reaction B', 'kcat_s_inv': 50.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('acon_C_c', 0.10, 0.10)], 'uniprot_accession_or_query': 'query: aconitase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'ICDHyr': {'pathway': 'TCA', 'ec_number': '1.1.1.42', 'enzyme': 'Isocitrate dehydrogenase (NADP)', 'kcat_s_inv': 60.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('icit_c', 0.05, 0.10), substrate('nadp_c', 0.05, 0.50)], 'uniprot_accession_or_query': 'query: icd isocitrate dehydrogenase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'AKGDH': {'pathway': 'TCA', 'ec_number': '1.2.4.2', 'enzyme': '2-oxoglutarate dehydrogenase complex', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('akg_c', 0.10, 0.20), substrate('coa_c', 0.05, 0.20), substrate('nad_c', 0.10, 1.00)], 'uniprot_accession_or_query': 'query: odh 2-oxoglutarate dehydrogenase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'AKGDa': {'pathway': 'TCA', 'ec_number': '1.2.4.2', 'enzyme': '2-oxoglutarate dehydrogenase E1 component', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('akg_c', 0.10, 0.20)], 'uniprot_accession_or_query': 'query: odhA 2-oxoglutarate dehydrogenase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'AKGDb': {'pathway': 'TCA', 'ec_number': '2.3.1.61', 'enzyme': 'Dihydrolipoyllysine-residue succinyltransferase', 'kcat_s_inv': 20.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('sdhlam_c', 0.10, 0.10), substrate('coa_c', 0.05, 0.20)], 'uniprot_accession_or_query': 'query: odhB succinyltransferase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'SUCOAS': {'pathway': 'TCA', 'ec_number': '6.2.1.5', 'enzyme': 'Succinate-CoA ligase', 'kcat_s_inv': 40.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('succ_c', 0.10, 0.20), substrate('atp_c', 0.10, 2.00)], 'uniprot_accession_or_query': 'query: sucC sucD succinate-CoA ligase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'SUCDi': {'pathway': 'TCA', 'ec_number': '1.3.5.1', 'enzyme': 'Succinate dehydrogenase', 'kcat_s_inv': 100.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('succ_c', 0.10, 0.20), substrate('q8_c', 0.05, 0.10)], 'uniprot_accession_or_query': 'query: sdhABCD succinate dehydrogenase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'FUM': {'pathway': 'TCA', 'ec_number': '4.2.1.2', 'enzyme': 'Fumarase', 'kcat_s_inv': 300.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('fum_c', 0.10, 0.20)], 'uniprot_accession_or_query': 'query: fum fumarate hydratase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'MDH': {'pathway': 'TCA', 'ec_number': '1.1.1.37', 'enzyme': 'Malate dehydrogenase', 'kcat_s_inv': 500.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('mal__L_c', 0.10, 0.20), substrate('nad_c', 0.10, 1.00)], 'uniprot_accession_or_query': 'query: mdh malate dehydrogenase Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
    'ME2': {'pathway': 'TCA', 'ec_number': '1.1.1.40', 'enzyme': 'NADP-dependent malic enzyme', 'kcat_s_inv': 100.0, 'enzyme_umol_per_gdw': 0.01, 'substrates': [substrate('mal__L_c', 0.10, 0.20), substrate('nadp_c', 0.05, 0.50)], 'uniprot_accession_or_query': 'query: maeB NADP-dependent malic enzyme Rhodopseudomonas palustris', 'kinetic_source': 'UniProtKB-compatible parameter slot'},
}



KINETIC_REFERENCE_METADATA = {
    'RBPC': ('Q07N61', 'R. palustris BisA53 RuBisCO homolog', 'Borrowed published/curated RuBisCO parameter slot; no DSM123-specific UniProt kinetic entry was available'),
    'RBCh': ('Q07N61', 'R. palustris BisA53 RuBisCO homolog', 'Borrowed RuBisCO oxygenase-side parameter slot; no DSM123-specific UniProt kinetic entry was available'),
    'RBPCcx': ('Q07N61', 'R. palustris BisA53 RuBisCO homolog', 'Borrowed RuBisCO mixed carboxylase/oxygenase parameter slot; no DSM123-specific UniProt kinetic entry was available'),
    'FDH': ('Q6NBU4', 'R. palustris CGA009 homolog; RpFDH DSM123 kinetics from this study', 'kcat=4.87 s^-1 and Km(formate)=237.8 uM (0.2378 mM) were taken from the RpFDH table supplied for this study; Q6NBU4 is a CGA009 homolog accession'),
    'RPE': ('Q6N379', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'FBA': ('Q6N0W8;Q6NB88', 'R. palustris CGA009 homologs', 'Borrowed homolog/literature parameter slot'),
    'FBA3': ('Q6N0W8;Q6NB88', 'R. palustris CGA009 FBA homologs', 'Borrowed aldolase-family parameter slot'),
    'PGK': ('P62419', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'GAPD': ('Q6NB84', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'TKT1': ('Q6NB83;A0AAE9Y5P7', 'R. palustris CGA009 homologs', 'Borrowed homolog/literature parameter slot'),
    'TKT2': ('Q6NB83;A0AAE9Y5P7', 'R. palustris CGA009 homologs', 'Borrowed homolog/literature parameter slot'),
    'TALA': ('A0AAE9Y0U7', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'SBP': ('not resolved', 'enzyme-class literature reference', 'No unique R. palustris UniProt accession was resolved for EC 3.1.3.37; parameter remains editable'),
    'FBP': ('Q6N0W5', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'CS': ('Q6N5R4', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'ACONT': ('Q6NDA7', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'ACONTa': ('Q6NDA7', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'ACONTb': ('Q6NDA7', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'ICDHyr': ('Q6N360', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'AKGDH': ('Q6NDC0;Q6NDC3', 'R. palustris CGA009 OGDH-complex homologs', 'Borrowed complex-level parameter slot'),
    'AKGDa': ('not resolved', 'enzyme-class literature reference', 'No unique R. palustris UniProt accession was resolved for the E1 component; parameter remains editable'),
    'AKGDb': ('Q6NDC0', 'R. palustris CGA009 OdhB homolog', 'Borrowed homolog/literature parameter slot'),
    'SUCOAS': ('Q6NDB7;Q6NDB8', 'R. palustris CGA009 homologs', 'Borrowed homolog/literature parameter slot'),
    'SUCDi': ('Q6ND92;Q6ND93', 'R. palustris CGA009 homologs', 'Borrowed homolog/literature parameter slot'),
    'FUM': ('Q6NA57', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
    'MDH': ('Q07UX5', 'R. palustris BisA53 homolog', 'Borrowed homolog/literature parameter slot'),
    'ME2': ('A0AAF0BQ94', 'R. palustris CGA009 homolog', 'Borrowed homolog/literature parameter slot'),
}

for _rid, _meta in KINETIC_REFERENCE_METADATA.items():
    if _rid in KINETIC_ASSUMPTIONS:
        KINETIC_ASSUMPTIONS[_rid]['kinetic_reference_accession'] = _meta[0]
        KINETIC_ASSUMPTIONS[_rid]['kinetic_reference_organism_strain'] = _meta[1]
        KINETIC_ASSUMPTIONS[_rid]['kinetic_reference_note'] = _meta[2]

# Only reactions with experimentally traceable kinetic constants are used as enzyme constraints.
# The other CBB/TCA rows remain in the table for mapping/provenance, but their placeholder kcat/Km values are not used.
for _rid, _kinetic in KINETIC_ASSUMPTIONS.items():
    if _rid == 'FDH':
        _kinetic['kinetic_record_status'] = 'verified: RpFDH this study; CGA009 UniProt Q6NBU4 used as homolog accession only'
        _kinetic['use_mm_constraint'] = True
    else:
        _kinetic['kinetic_record_status'] = 'not constrained: real kcat/Km record not verified for this reaction/accession'
        _kinetic['use_mm_constraint'] = False

CURATED_REAL_KINETICS = {
    'CS': {
        'kcat_s_inv': 580.0,
        'substrates': [substrate('accoa_c', 0.060, 0.20, 'curated Km from R. capsulatus CS'), substrate('oaa_c', 0.030, 0.05, 'curated Km from R. capsulatus CS')],
        'kinetic_reference_accession': 'BRENDA EC 2.3.3.1; literature J. Bacteriol. 1990',
        'kinetic_reference_organism_strain': 'R. capsulatus',
        'kinetic_reference_note': 'Curated representative citrate synthase kinetics from Gram-negative purple bacterium R. capsulatus; acetyl-CoA Km=60 uM, OAA Km=30 uM, kcat=580 s^-1.',
    },
    'ACONT': {
        'kcat_s_inv': 35.0,
        'substrates': [substrate('cit_c', 1.2, 0.20, 'E. coli K-12 AcnA/AcnB homolog proxy'), substrate('icit_c', 0.4, 0.10, 'E. coli K-12 AcnA/AcnB homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 4.2.1.3; E. coli K-12 AcnA/AcnB homolog proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated aconitase homolog proxy because purified purple-bacterial absolute kinetic data are limited; citrate Km=1.2 mM, isocitrate Km=0.4 mM, kcat=35 s^-1.',
    },
    'ACONTa': {
        'kcat_s_inv': 35.0,
        'substrates': [substrate('cit_c', 1.2, 0.20, 'E. coli K-12 AcnA/AcnB homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 4.2.1.3; E. coli K-12 AcnA/AcnB homolog proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated aconitase half-reaction A proxy; citrate Km=1.2 mM, kcat=35 s^-1.',
    },
    'ACONTb': {
        'kcat_s_inv': 35.0,
        'substrates': [substrate('icit_c', 0.4, 0.10, 'E. coli K-12 AcnA/AcnB homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 4.2.1.3; E. coli K-12 AcnA/AcnB homolog proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated aconitase half-reaction B proxy; isocitrate Km=0.4 mM, kcat=35 s^-1.',
    },
    'ICDHyr': {
        'kcat_s_inv': 78.0,
        'substrates': [substrate('icit_c', 0.016, 0.10, 'R. rubrum ICDH'), substrate('nadp_c', 0.011, 0.50, 'R. rubrum ICDH')],
        'kinetic_reference_accession': 'BRENDA/SABIO-RK EC 1.1.1.42; J. Biol. Chem. 1988',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated purple-bacterial NADP-dependent ICDH kinetics; isocitrate Km=16 uM, NADP+ Km=11 uM, kcat=78 s^-1.',
    },
    'AKGDH': {
        'kcat_s_inv': 85.0,
        'substrates': [substrate('akg_c', 0.15, 0.20, 'R. rubrum OGDH E1-limited complex proxy'), substrate('nad_c', 0.10, 1.00, 'R. rubrum OGDH E1-limited complex proxy')],
        'kinetic_reference_accession': 'BRENDA EC 1.2.4.2; E1-limited OGDH complex proxy',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated alpha-ketoglutarate dehydrogenase complex proxy using E1-limited activity; alpha-KG Km=0.15 mM, NAD+ Km~0.1 mM, kcat=85 s^-1.',
    },
    'AKGDa': {
        'kcat_s_inv': 85.0,
        'substrates': [substrate('akg_c', 0.15, 0.20, 'R. rubrum OGDH E1-limited proxy')],
        'kinetic_reference_accession': 'BRENDA EC 1.2.4.2; E1-limited OGDH proxy',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated E1-limited alpha-ketoglutarate dehydrogenase half-reaction proxy; alpha-KG Km=0.15 mM, kcat=85 s^-1.',
    },
    'AKGDb': {
        'kcat_s_inv': 85.0,
        'substrates': [substrate('akg_c', 0.15, 0.20, 'R. rubrum OGDH complex-level E1-limited proxy')],
        'kinetic_reference_accession': 'BRENDA EC 1.2.4.2; complex-level proxy applied to AKGDb',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated complex-level OGDH proxy applied to model subreaction AKGDb to keep the split reaction consistent with the E1-limited complex capacity.',
    },
    'SUCOAS': {
        'kcat_s_inv': 50.0,
        'substrates': [substrate('succ_c', 1.5, 0.20, 'E. coli/model-organism SCS proxy'), substrate('atp_c', 0.20, 2.00, 'E. coli/model-organism SCS proxy')],
        'kinetic_reference_accession': 'BRENDA EC 6.2.1.5; E. coli/model-organism proxy',
        'kinetic_reference_organism_strain': 'E. coli / model organism',
        'kinetic_reference_note': 'Curated succinate-CoA ligase proxy because purple-bacterial parameters are missing; succinate Km~1.5 mM, ATP Km=0.2 mM, kcat~50 s^-1.',
    },
    'SUCDi': {
        'kcat_s_inv': 60.0,
        'substrates': [substrate('succ_c', 0.45, 0.20, 'R. sphaeroides SDH membrane-complex proxy')],
        'kinetic_reference_accession': 'BRENDA EC 1.3.5.1/1.3.99.1; Biochim. Biophys. Acta SDH complex studies',
        'kinetic_reference_organism_strain': 'R. sphaeroides',
        'kinetic_reference_note': 'Curated membrane-bound succinate dehydrogenase complex proxy; succinate Km=0.45 mM, kcat=60 s^-1.',
    },
    'FUM': {
        'kcat_s_inv': 600.0,
        'substrates': [substrate('fum_c', 0.15, 0.20, 'E. coli K-12 FumA/B Class I homolog proxy'), substrate('mal__L_c', 0.45, 0.20, 'E. coli K-12 FumA/B Class I homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 4.2.1.2; E. coli K-12 FumA/B proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated conserved fumarase proxy; fumarate Km=0.15 mM, malate Km=0.45 mM, kcat~600 s^-1.',
    },
    'MDH': {
        'kcat_s_inv': 450.0,
        'substrates': [substrate('oaa_c', 0.038, 0.05, 'R. capsulatus MDH'), substrate('nadh_c', 0.019, 0.10, 'R. capsulatus MDH')],
        'kinetic_reference_accession': 'BRENDA EC 1.1.1.37; Arch. Microbiol. R. capsulatus MDH',
        'kinetic_reference_organism_strain': 'R. capsulatus',
        'kinetic_reference_note': 'Curated purple-bacterial malate dehydrogenase kinetics; OAA Km=38 uM, NADH Km=19 uM, kcat=450 s^-1.',
    },
    'ME2': {
        'kcat_s_inv': 35.0,
        'substrates': [substrate('mal__L_c', 0.28, 0.20, 'R. rubrum NADP-dependent malic enzyme'), substrate('nadp_c', 0.030, 0.50, 'R. rubrum NADP-dependent malic enzyme')],
        'kinetic_reference_accession': 'BRENDA EC 1.1.1.40; R. rubrum malic enzyme',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated R. rubrum NADP-dependent malic enzyme kinetics; malate Km=0.28 mM, NADP+ Km=0.03 mM, kcat=35 s^-1.',
    },
}

for _rid, _curated in CURATED_REAL_KINETICS.items():
    if _rid in KINETIC_ASSUMPTIONS:
        KINETIC_ASSUMPTIONS[_rid].update(_curated)
        KINETIC_ASSUMPTIONS[_rid]['kinetic_record_status'] = 'curated: user-provided BRENDA/SABIO-RK/literature representative kinetic record'
        KINETIC_ASSUMPTIONS[_rid]['use_mm_constraint'] = True

CURATED_CBB_KINETICS = {
    'RBPC': {
        'kcat_s_inv': 2.4,
        'substrates': [substrate('co2_c', 0.014, 0.01, 'R. rubrum Form II RuBisCO curated Km'), substrate('rb15bp_c', 0.015, 0.10, 'R. rubrum Form II RuBisCO curated Km')],
        'kinetic_reference_accession': 'SABIO-RK/BRENDA EC 4.1.1.39; Tabita MMBR 2008 summary',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated purple-bacterial Form II RuBisCO carboxylation kinetics; CO2 Km=14 uM, RuBP Km=15 uM, kcat=2.4 s^-1.',
    },
    'RBCh': {
        'kcat_s_inv': 0.5,
        'substrates': [substrate('o2_c', 0.300, 0.25, 'R. rubrum RuBisCO oxygenase curated Km'), substrate('rb15bp_c', 0.015, 0.10, 'R. rubrum RuBisCO oxygenase curated Km')],
        'kinetic_reference_accession': 'BRENDA EC 4.1.1.39; J. Biol. Chem. 1980 oxygenase characterization',
        'kinetic_reference_organism_strain': 'R. rubrum',
        'kinetic_reference_note': 'Curated RuBisCO oxygenase side reaction kinetics; O2 Km~300 uM, RuBP Km=15 uM, kcat~0.5 s^-1.',
    },
    'RBPCcx': {
        'kcat_s_inv': 2.2,
        'substrates': [substrate('co2_c', 0.017, 0.01, 'R. palustris/R. rubrum apparent RuBisCO proxy'), substrate('rb15bp_c', 0.015, 0.10, 'apparent RuBisCO proxy')],
        'kinetic_reference_accession': 'curated network-modeling proxy for EC 4.1.1.39',
        'kinetic_reference_organism_strain': 'R. palustris / R. rubrum',
        'kinetic_reference_note': 'Curated apparent activated-state RuBisCO proxy for simplified models; CO2 Km=14-20 uM represented as 17 uM, kcat=2.0-2.4 s^-1 represented as 2.2 s^-1.',
    },
    'RPE': {
        'kcat_s_inv': 85.0,
        'substrates': [substrate('ru5p__D_c', 0.35, 0.10, 'E. coli K-12 RPE homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 5.1.3.1; E. coli K-12 homolog proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated RPE homolog proxy because purified purple-bacterial absolute kinetics are limited; Ru5P Km=0.35 mM, kcat=85 s^-1.',
    },
    'FBA': {
        'kcat_s_inv': 28.0,
        'substrates': [substrate('fdp_c', 0.015, 0.10, 'Gram-negative Class II aldolase curated Km')],
        'kinetic_reference_accession': 'BRENDA EC 4.1.2.13; Eur. J. Biochem. 2000 Class II aldolase',
        'kinetic_reference_organism_strain': 'Gram-negative bacterium Class II aldolase',
        'kinetic_reference_note': 'Curated conserved Class II fructose-bisphosphate aldolase kinetics; FBP Km=15 uM, kcat=28 s^-1.',
    },
    'FBA3': {
        'kcat_s_inv': 18.0,
        'substrates': [substrate('s17bp_c', 0.035, 0.10, 'Gram-negative Class II aldolase SBP curated Km')],
        'kinetic_reference_accession': 'BRENDA EC 4.1.2.13; Eur. J. Biochem. 2000 Class II aldolase',
        'kinetic_reference_organism_strain': 'Gram-negative bacterium Class II aldolase',
        'kinetic_reference_note': 'Curated Class II aldolase sedoheptulose-bisphosphate activity; SBP Km=35 uM, kcat=18 s^-1.',
    },
    'PGK': {
        'kcat_s_inv': 250.0,
        'substrates': [substrate('3pg_c', 0.65, 0.10, 'E. coli/Rhodobacter PGK curated Km'), substrate('atp_c', 0.18, 2.00, 'E. coli/Rhodobacter PGK curated Km')],
        'kinetic_reference_accession': 'BRENDA EC 2.7.2.3; J. Biol. Chem. standard PGK kinetics',
        'kinetic_reference_organism_strain': 'E. coli / Rhodobacter',
        'kinetic_reference_note': 'Curated conserved phosphoglycerate kinase kinetics; 3-PGA Km=0.65 mM, ATP Km=0.18 mM, kcat=250 s^-1.',
    },
    'GAPD': {
        'kcat_s_inv': 55.0,
        'substrates': [substrate('13dpg_c', 0.065, 0.10, 'R. sphaeroides GAPDH curated Km'), substrate('nadh_c', 0.030, 0.10, 'R. sphaeroides GAPDH curated Km')],
        'kinetic_reference_accession': 'BRENDA EC 1.2.1.12; Arch. Microbiol. 1993 R. sphaeroides GAPDH',
        'kinetic_reference_organism_strain': 'R. sphaeroides',
        'kinetic_reference_note': 'Curated purple-bacterial GAPDH NAD(P)H preference data; 1,3-BPG Km=65 uM, NADH Km=30 uM, kcat=55 s^-1.',
    },
    'TKT1': {
        'kcat_s_inv': 45.0,
        'substrates': [substrate('r5p_c', 0.18, 0.10, 'E. coli K-12 TktA homolog proxy'), substrate('xu5p__D_c', 0.12, 0.10, 'E. coli K-12 TktA homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 2.2.1.1; E. coli K-12 TktA proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated transketolase R5P+X5P proxy; R5P Km=0.18 mM, X5P Km=0.12 mM, kcat=45 s^-1.',
    },
    'TKT2': {
        'kcat_s_inv': 40.0,
        'substrates': [substrate('e4p_c', 0.08, 0.10, 'E. coli K-12 TktA homolog proxy'), substrate('xu5p__D_c', 0.12, 0.10, 'E. coli K-12 TktA homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 2.2.1.1; E. coli K-12 TktA proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated transketolase E4P+X5P proxy; E4P Km=0.08 mM, X5P Km=0.12 mM, kcat=40 s^-1.',
    },
    'TALA': {
        'kcat_s_inv': 35.0,
        'substrates': [substrate('s7p_c', 0.15, 0.10, 'E. coli K-12 TalB homolog proxy'), substrate('g3p_c', 0.09, 0.10, 'E. coli K-12 TalB homolog proxy')],
        'kinetic_reference_accession': 'BRENDA EC 2.2.1.2; E. coli K-12 TalB proxy',
        'kinetic_reference_organism_strain': 'E. coli K-12',
        'kinetic_reference_note': 'Curated transaldolase homolog proxy; S7P Km=0.15 mM, GAP Km=0.09 mM, kcat=35 s^-1.',
    },
    'SBP': {
        'kcat_s_inv': 12.0,
        'substrates': [substrate('s17bp_c', 0.025, 0.10, 'Rhodobacter GlpX-family FBPase/SBPase proxy')],
        'kinetic_reference_accession': 'SABIO-RK/literature EC 3.1.3.37; MMBR 2014 GlpX-family review',
        'kinetic_reference_organism_strain': 'Rhodobacter GlpX family',
        'kinetic_reference_note': 'Curated GlpX-family sedoheptulose-bisphosphatase proxy; SBP Km~25 uM, kcat=12 s^-1.',
    },
    'FBP': {
        'kcat_s_inv': 18.0,
        'substrates': [substrate('fdp_c', 0.012, 0.10, 'Rhodobacter GlpX-family FBPase/SBPase proxy')],
        'kinetic_reference_accession': 'SABIO-RK/literature EC 3.1.3.11; MMBR 2014 GlpX-family review',
        'kinetic_reference_organism_strain': 'Rhodobacter GlpX family',
        'kinetic_reference_note': 'Curated GlpX-family fructose-bisphosphatase proxy; FBP Km=12 uM, kcat=18 s^-1.',
    },
}

for _rid, _curated in CURATED_CBB_KINETICS.items():
    if _rid in KINETIC_ASSUMPTIONS:
        KINETIC_ASSUMPTIONS[_rid].update(_curated)
        KINETIC_ASSUMPTIONS[_rid]['kinetic_record_status'] = 'curated: user-provided BRENDA/SABIO-RK/literature representative kinetic record'
        KINETIC_ASSUMPTIONS[_rid]['use_mm_constraint'] = True

def build_mm_kinetics_table(model):
    rpal_map = _load_rpal_homolog_map()
    product_map = _load_dsm_product_map()
    rows = []
    substrate_rows = []
    for rid, kinetic in KINETIC_ASSUMPTIONS.items():
        if rid not in model.reactions:
            continue
        rxn = model.reactions.get_by_id(rid)
        locus_tags = kinetic.get('dsm_locus_tags_override') or _extract_locus_tags(rxn.gene_reaction_rule)
        products, protein_ids, aa_lengths = [], [], []
        for locus in locus_tags:
            ann = product_map.get(locus, {})
            if ann:
                products.append(f"{locus}:{ann.get('product', '')}")
                protein_ids.append(str(ann.get('protein_id', '')))
                aa_lengths.append(str(ann.get('aa_length', '')))
        rpal_homologs = ';'.join(sorted(set(rpal_map.get(locus, '') for locus in locus_tags if rpal_map.get(locus, ''))))
        vmax_mmol_gdw_h = kinetic['kcat_s_inv'] * (kinetic['enzyme_umol_per_gdw'] / 1000.0) * 3600.0
        rows.append({
            'reaction_id': rid,
            'reaction_name': rxn.name,
            'pathway': kinetic['pathway'],
            'ec_number': kinetic['ec_number'],
            'enzyme_name': kinetic['enzyme'],
            'dsm_locus_tags': ';'.join(locus_tags),
            'dsm_products': ';'.join(products),
            'dsm_protein_ids': ';'.join(protein_ids),
            'dsm_aa_lengths': ';'.join(aa_lengths),
            'rpal_homolog_locus_tags': rpal_homologs,
            'uniprot_accession_or_query': kinetic['uniprot_accession_or_query'],
            'kinetic_source': kinetic['kinetic_source'],
            'kinetic_reference_accession': kinetic.get('kinetic_reference_accession', kinetic.get('uniprot_accession_or_query', 'not resolved')),
            'kinetic_reference_organism_strain': kinetic.get('kinetic_reference_organism_strain', 'not resolved'),
            'kinetic_reference_note': kinetic.get('kinetic_reference_note', kinetic['kinetic_source']),
            'kinetic_record_status': kinetic.get('kinetic_record_status', 'not verified'),
            'use_mm_constraint': kinetic.get('use_mm_constraint', False),
            'kcat_s_inv': kinetic['kcat_s_inv'],
            'pgidnb_locus_tags': kinetic.get('pgidnb_locus_tags', ''),
            'rnaseq_6h_median_expression': kinetic.get('rnaseq_6h_median_expression', np.nan),
            'rnaseq_total_expression_for_constrained_reactions': kinetic.get('rnaseq_total_expression_for_constrained_reactions', np.nan),
            'rnaseq_allocation_fraction': kinetic.get('rnaseq_allocation_fraction', np.nan),
            'model_biomass_protein_fraction_g_per_gdw': kinetic.get('model_biomass_protein_fraction_g_per_gdw', np.nan),
            'enzyme_pool_fraction_of_biomass_protein': kinetic.get('enzyme_pool_fraction_of_biomass_protein', np.nan),
            'estimated_enzyme_mw_kda': kinetic.get('estimated_enzyme_mw_kda', np.nan),
            'allocated_enzyme_mass_g_per_gdw': kinetic.get('allocated_enzyme_mass_g_per_gdw', np.nan),
            'enzyme_umol_per_gdw': kinetic['enzyme_umol_per_gdw'],
            'vmax_mmol_gdw_h': vmax_mmol_gdw_h,
            'mm_formula': 'v = kcat_s_inv * (enzyme_umol_per_gdw / 1000) * 3600 * min(S_i / (Km_i + S_i))',
            'substrates': ';'.join(s['metabolite_id'] for s in kinetic['substrates']),
            'gene_reaction_rule': rxn.gene_reaction_rule,
        })
        for s in kinetic['substrates']:
            substrate_rows.append({
                'reaction_id': rid,
                'metabolite_id': s['metabolite_id'],
                'km_mM': s['km_mM'],
                'default_concentration_mM': s['default_mM'],
                'concentration_source': s['source'],
            })
    return pd.DataFrame(rows), pd.DataFrame(substrate_rows)


def _biomass_protein_fraction_g_per_gdw(model):
    """Read the protein mass fraction directly from the biomass reaction."""
    if 'BIOMASS__1' not in model.reactions:
        return 0.5112
    biomass = model.reactions.get_by_id('BIOMASS__1')
    for metabolite, coefficient in biomass.metabolites.items():
        if metabolite.id == 'bm_pro_c':
            return abs(float(coefficient))
    return 0.5112


def _estimate_enzyme_mw_kda(locus_tags, product_map):
    """Approximate enzyme molecular weight from GPR-associated protein lengths."""
    aa_lengths = []
    for locus in locus_tags:
        ann = product_map.get(locus, {})
        value = ann.get('aa_length', np.nan)
        if pd.notna(value):
            aa_lengths.append(float(value))
    if not aa_lengths:
        return 50.0
    mw_kda = sum(aa_lengths) * 0.110
    return mw_kda if mw_kda > 0 else 50.0


def apply_rnaseq_enzyme_scaling(model, rnaseq_path=None, enzyme_pool_fraction_of_biomass_protein=1.0):
    """Allocate the biomass protein pool to constrained reactions using RNA-seq relative abundance."""
    if rnaseq_path is None:
        rnaseq_path = MM_DFBA_OUTPUT_DIR / 'Supplementary File S1. DSM 123 RNA-seq DEG.xlsx'
    rnaseq_path = Path(rnaseq_path)
    mapping_path = Path('DSM123_blast_gene_rename_output/gene_mapping_consensus_to_DSM123.csv')
    product_map = _load_dsm_product_map()
    protein_fraction_g_per_gdw = _biomass_protein_fraction_g_per_gdw(model)
    rows = []
    if not rnaseq_path.exists() or not mapping_path.exists():
        for rid, kinetic in KINETIC_ASSUMPTIONS.items():
            if rid not in model.reactions:
                continue
            locus_tags = _extract_locus_tags(model.reactions.get_by_id(rid).gene_reaction_rule)
            mw_kda = _estimate_enzyme_mw_kda(locus_tags, product_map)
            allocated_mass = protein_fraction_g_per_gdw * enzyme_pool_fraction_of_biomass_protein / max(len(KINETIC_ASSUMPTIONS), 1)
            enzyme_umol = allocated_mass / (mw_kda * 1000.0) * 1e6
            kinetic['rnaseq_6h_median_expression'] = np.nan
            kinetic['rnaseq_allocation_fraction'] = 1.0 / max(len(KINETIC_ASSUMPTIONS), 1)
            kinetic['model_biomass_protein_fraction_g_per_gdw'] = protein_fraction_g_per_gdw
            kinetic['enzyme_pool_fraction_of_biomass_protein'] = enzyme_pool_fraction_of_biomass_protein
            kinetic['estimated_enzyme_mw_kda'] = mw_kda
            kinetic['allocated_enzyme_mass_g_per_gdw'] = allocated_mass
            kinetic['enzyme_umol_per_gdw'] = enzyme_umol
        return pd.DataFrame()

    mapping = pd.read_csv(mapping_path)
    dsm_to_pgid = (
        mapping.dropna(subset=['subject_locus_tag', 'query_locus_tag'])
        .groupby('subject_locus_tag')['query_locus_tag']
        .apply(lambda x: sorted(set(map(str, x))))
        .to_dict()
    )
    rnaseq = pd.read_excel(rnaseq_path, sheet_name='123 0h vs 6h')
    expr_cols = [c for c in rnaseq.columns if str(c).endswith('-6h')]
    rnaseq['rnaseq_6h_mean'] = rnaseq[expr_cols].mean(axis=1)
    expr = dict(zip(rnaseq['ID'].astype(str), rnaseq['rnaseq_6h_mean']))

    for rid, kinetic in KINETIC_ASSUMPTIONS.items():
        if rid not in model.reactions:
            continue
        locus_tags = kinetic.get('dsm_locus_tags_override') or _extract_locus_tags(model.reactions.get_by_id(rid).gene_reaction_rule)
        pgids, values = [], []
        override_pgids = kinetic.get('rnaseq_locus_tags_override')
        if override_pgids:
            pgids = list(override_pgids)
            for pgid in pgids:
                value = expr.get(pgid, np.nan)
                if pd.notna(value):
                    values.append(float(value))
        else:
            for locus in locus_tags:
                for pgid in dsm_to_pgid.get(locus, []):
                    pgids.append(pgid)
                    value = expr.get(pgid, np.nan)
                    if pd.notna(value):
                        values.append(float(value))
        score = float(np.median(values)) if values else np.nan
        mw_kda = _estimate_enzyme_mw_kda(locus_tags, product_map)
        rows.append({
            'reaction_id': rid,
            'pgidnb_locus_tags': ';'.join(sorted(set(pgids))),
            'rnaseq_6h_median_expression': score,
            'estimated_enzyme_mw_kda': mw_kda,
        })

    scaling_df = pd.DataFrame(rows)
    total_score = float(scaling_df['rnaseq_6h_median_expression'].dropna().sum())
    if total_score <= 0:
        total_score = float(len(scaling_df))
    total_enzyme_mass_g_per_gdw = protein_fraction_g_per_gdw * enzyme_pool_fraction_of_biomass_protein

    for _, row in scaling_df.iterrows():
        rid = row['reaction_id']
        score = row['rnaseq_6h_median_expression']
        allocation = float(score / total_score) if pd.notna(score) and total_score > 0 else 1.0 / max(len(scaling_df), 1)
        allocated_mass = total_enzyme_mass_g_per_gdw * allocation
        mw_kda = float(row['estimated_enzyme_mw_kda'])
        if mw_kda <= 0:
            mw_kda = 50.0
        enzyme_umol = allocated_mass / (mw_kda * 1000.0) * 1e6
        KINETIC_ASSUMPTIONS[rid]['pgidnb_locus_tags'] = row['pgidnb_locus_tags']
        KINETIC_ASSUMPTIONS[rid]['rnaseq_6h_median_expression'] = score
        KINETIC_ASSUMPTIONS[rid]['rnaseq_total_expression_for_constrained_reactions'] = total_score
        KINETIC_ASSUMPTIONS[rid]['rnaseq_allocation_fraction'] = allocation
        KINETIC_ASSUMPTIONS[rid]['model_biomass_protein_fraction_g_per_gdw'] = protein_fraction_g_per_gdw
        KINETIC_ASSUMPTIONS[rid]['enzyme_pool_fraction_of_biomass_protein'] = enzyme_pool_fraction_of_biomass_protein
        KINETIC_ASSUMPTIONS[rid]['estimated_enzyme_mw_kda'] = mw_kda
        KINETIC_ASSUMPTIONS[rid]['allocated_enzyme_mass_g_per_gdw'] = allocated_mass
        KINETIC_ASSUMPTIONS[rid]['enzyme_umol_per_gdw'] = enzyme_umol
    return scaling_df


def _concentration_map(formate_mM, co2_mM):
    concentration = {}
    for kinetic in KINETIC_ASSUMPTIONS.values():
        for s in kinetic['substrates']:
            concentration[s['metabolite_id']] = s['default_mM']
    concentration['for_e'] = max(formate_mM, 0.0)
    concentration['for_c'] = max(formate_mM, 0.0)
    concentration['co2_c'] = max(0.01, 0.01 + max(co2_mM, 0.0))
    concentration['o2_c'] = 0.25
    return concentration


def _mm_effective_bound(kinetic, concentration):
    vmax = kinetic['kcat_s_inv'] * (kinetic['enzyme_umol_per_gdw'] / 1000.0) * 3600.0
    factors = []
    details = []
    for s in kinetic['substrates']:
        substrate_conc = float(concentration.get(s['metabolite_id'], s['default_mM']))
        km = float(s['km_mM'])
        factor = substrate_conc / (km + substrate_conc) if substrate_conc > 0 else 0.0
        factors.append(factor)
        details.append(f"{s['metabolite_id']}:S={substrate_conc:.6g},Km={km:.6g},sat={factor:.6g}")
    limiting_factor = min(factors) if factors else 1.0
    return vmax * limiting_factor, vmax, limiting_factor, ';'.join(details)


def apply_mm_constraints(model, concentration, time_h, mechanism, initial_formate_mM):
    records = []
    for rid, kinetic in KINETIC_ASSUMPTIONS.items():
        if rid not in model.reactions or not kinetic.get('use_mm_constraint', False):
            continue
        rxn = model.reactions.get_by_id(rid)
        mm_bound, vmax, limiting_factor, details = _mm_effective_bound(kinetic, concentration)
        old_lb, old_ub = rxn.lower_bound, rxn.upper_bound
        if old_lb < 0 and old_ub > 0:
            rxn.lower_bound = max(old_lb, -mm_bound)
            rxn.upper_bound = min(old_ub, mm_bound)
        elif old_ub <= 0:
            rxn.lower_bound = max(old_lb, -mm_bound)
            rxn.upper_bound = min(old_ub, 0.0)
        else:
            rxn.lower_bound = max(old_lb, 0.0)
            rxn.upper_bound = min(old_ub, mm_bound)
        records.append({
            'time_h': time_h,
            'mechanism': mechanism,
            'initial_formate_mM': initial_formate_mM,
            'reaction_id': rid,
            'pathway': kinetic['pathway'],
            'vmax_mmol_gdw_h': vmax,
            'mm_effective_bound_mmol_gdw_h': mm_bound,
            'limiting_saturation_factor': limiting_factor,
            'substrate_terms': details,
            'old_lower_bound': old_lb,
            'old_upper_bound': old_ub,
            'new_lower_bound': rxn.lower_bound,
            'new_upper_bound': rxn.upper_bound,
        })
    return records


def run_mm_enzyme_constrained_dfba(model_path=MODEL_JSON, output_dir=MM_DFBA_OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    base_model = cobra.io.load_json_model(str(model_path))
    base_model = apply_base_conditions(base_model)
    base_model.objective = 'BIOMASS__1'

    rnaseq_scaling_df = apply_rnaseq_enzyme_scaling(base_model, output_dir / 'Supplementary File S1. DSM 123 RNA-seq DEG.xlsx', enzyme_pool_fraction_of_biomass_protein=1.0)
    rnaseq_scaling_df.to_csv(output_dir / 'rnaseq_allocation.csv', index=False, encoding='utf-8-sig')
    kinetics_df, substrate_df = build_mm_kinetics_table(base_model)
    kinetics_df.to_csv(output_dir / 'kinetics.csv', index=False, encoding='utf-8-sig')
    substrate_df.to_csv(output_dir / 'substrates.csv', index=False, encoding='utf-8-sig')

    t_max, dt = 360, 1.0
    x0 = 0.02
    initial_formates = [5, 10, 20, 50]
    mechanisms = ['Active', 'Diffusion']
    vmax_transport = 1.786555464
    km_fort = 0.01814
    p_envelope = 1.0e-8
    area_spec_nichols = 1.32e5
    diff_conv = p_envelope * area_spec_nichols * 3600 / 1000
    pka_formic, ph_env = 3.75, 7.0
    total_light, max_specific_light = 50.0, 60.0

    all_results = {mech: {} for mech in mechanisms}
    records = []
    flux_records = []
    bound_records = []
    constrained_reactions = [rid for rid, kinetic in KINETIC_ASSUMPTIONS.items() if rid in base_model.reactions and kinetic.get('use_mm_constraint', False)]

    for mech in mechanisms:
        for init_for in initial_formates:
            x, formate, co2 = x0, float(init_for), 0.0
            history = {'time': [], 'X': [], 'For': [], 'CO2': []}
            for t in np.arange(0, t_max + dt, dt):
                if formate > 1e-4:
                    if mech == 'Active':
                        v_uptake = vmax_transport * (formate / (km_fort + formate))
                    else:
                        frac_undiss = 1.0 / (1.0 + 10 ** (ph_env - pka_formic))
                        v_uptake = diff_conv * (formate * frac_undiss)
                else:
                    v_uptake = 0.0

                concentration = _concentration_map(formate, co2)
                with base_model as m:
                    bound_records.extend(apply_mm_constraints(m, concentration, t, mech, init_for))
                    m.reactions.get_by_id('EX_for_e').lower_bound = -v_uptake
                    if 'EX_photon_purple_e' in m.reactions:
                        m.reactions.get_by_id('EX_photon_purple_e').lower_bound = -min(max_specific_light, total_light / max(x, 1e-12))

                    sol_value = m.slim_optimize()
                    if sol_value and sol_value > 1e-9:
                        full_sol = m.optimize()
                        mu = float(full_sol.objective_value)
                        f_for = float(full_sol.fluxes.get('EX_for_e', 0.0))
                        f_co2 = float(full_sol.fluxes.get('EX_co2_e', 0.0))
                        for rid in constrained_reactions:
                            flux_records.append({
                                'mechanism': mech,
                                'initial_formate_mM': init_for,
                                'time_h': t,
                                'reaction_id': rid,
                                'flux_mmol_gDW_h': float(full_sol.fluxes.get(rid, 0.0)),
                            })
                    else:
                        mu, f_for, f_co2 = 0.0, 0.0, 0.0

                history['time'].append(t)
                history['X'].append(x)
                history['For'].append(formate)
                history['CO2'].append(co2)
                records.append({
                    'mechanism': mech,
                    'initial_formate_mM': init_for,
                    'time_h': t,
                    'biomass_gDW_L': x,
                    'formate_mM': formate,
                    'co2_mM': co2,
                    'mu_h_inv': mu,
                    'formate_exchange_flux': f_for,
                    'co2_exchange_flux': f_co2,
                    'formate_uptake_capacity': v_uptake,
                    'co2_proxy_mM_for_mm': concentration['co2_c'],
                })
                x += mu * x * dt
                formate += f_for * x * dt
                co2 += f_co2 * x * dt
                if formate < 0:
                    formate = 0.0
            all_results[mech][init_for] = history

    time_df = pd.DataFrame(records)
    flux_df = pd.DataFrame(flux_records)
    bounds_df = pd.DataFrame(bound_records)
    summary_rows = []
    for mech in mechanisms:
        for init_for in initial_formates:
            h = all_results[mech][init_for]
            summary_rows.append({
                'mechanism': mech,
                'initial_formate_mM': init_for,
                'final_biomass_gDW_L': h['X'][-1],
                'final_formate_mM': h['For'][-1],
                'final_co2_mM': h['CO2'][-1],
            })
    summary_df = pd.DataFrame(summary_rows)

    provenance = (
        '# Michaelis-Menten enzyme constraint notes\n\n'
        'Each listed CBB/TCA reaction is constrained at every dFBA step with '
        '`v = kcat_s_inv * (enzyme_umol_per_gdw / 1000) * 3600 * min(S_i/(Km_i+S_i))`. '
        'The model has ACXYSJ locus tags and local GenBank-style protein IDs. Exact UniProt accessions were not present in the local annotation, so the exported table keeps a UniProt query field for each reaction. '
        'Replace kcat, Km, enzyme amount, and intracellular substrate assumptions with accession-specific UniProt/BRENDA/SABIO-RK values when those are confirmed.\n'
    )
    (output_dir / 'mm_kinetics_provenance_notes.md').write_text(provenance, encoding='utf-8')

    target_formate = 20
    plot_time = np.array(all_results['Active'][target_formate]['time'], dtype=float)
    plot_data = pd.DataFrame({
        'Time_h': plot_time,
        'Time_d': plot_time / 24.0,
        'Active_Biomass_gDW_L': all_results['Active'][target_formate]['X'],
        'Active_Formate_mM': all_results['Active'][target_formate]['For'],
        'Diffusion_Biomass_gDW_L': all_results['Diffusion'][target_formate]['X'],
        'Diffusion_Formate_mM': all_results['Diffusion'][target_formate]['For'],
    })
    plot_data.to_csv(output_dir / 'mm_dfba_20mM_active_diffusion_plot_data.csv', index=False)

    plt.rcParams['font.sans-serif'] = ['Arial']
    plt.rcParams['axes.unicode_minus'] = False
    fig, ax_biomass = plt.subplots(figsize=(9, 6))
    ax_formate = ax_biomass.twinx()
    colors = {'Active': '#c2185b', 'Diffusion': '#106e7e'}

    for mech in mechanisms:
        history = all_results[mech][target_formate]
        time_days = np.array(history['time'], dtype=float) / 24.0
        ax_biomass.plot(
            time_days,
            history['X'],
            color=colors[mech],
            linestyle='-',
            linewidth=3.0,
            label=f'{mech} transport biomass',
        )
        ax_formate.plot(
            time_days,
            history['For'],
            color=colors[mech],
            linestyle='--',
            linewidth=2.2,
            alpha=0.90,
            label=f'{mech} transport formate',
        )

    max_biomass = max(
        max(all_results['Active'][target_formate]['X']),
        max(all_results['Diffusion'][target_formate]['X']),
    )
    max_formate = max(
        max(all_results['Active'][target_formate]['For']),
        max(all_results['Diffusion'][target_formate]['For']),
        target_formate,
    )

    ax_biomass.set_title('20 mM formate dFBA: active transport vs diffusion', fontsize=15, fontweight='bold', pad=14)
    ax_biomass.set_xlabel('Time (d)', fontsize=13, fontweight='bold')
    ax_biomass.set_ylabel(r'Biomass (gDW L$^{-1}$)', fontsize=13, fontweight='bold')
    ax_formate.set_ylabel('Formate remaining (mM)', fontsize=13, fontweight='bold')
    ax_biomass.set_xlim(0, t_max / 24.0)
    ax_biomass.set_ylim(0, max(0.05, max_biomass * 1.12))
    ax_formate.set_ylim(-0.5, max_formate * 1.08)
    ax_biomass.grid(True, linestyle=':', alpha=0.45)

    lines_biomass, labels_biomass = ax_biomass.get_legend_handles_labels()
    lines_formate, labels_formate = ax_formate.get_legend_handles_labels()
    ax_biomass.legend(
        lines_biomass + lines_formate,
        labels_biomass + labels_formate,
        loc='upper left',
        frameon=True,
        fontsize=10,
    )

    plt.tight_layout()
    output_figure = output_dir / 'mm_enzyme_constrained_dfba_20mM_active_diffusion.svg'
    fig.savefig(output_figure, format='svg', dpi=300, bbox_inches='tight')
    plt.close(fig)

    sampling_days = [0, 2, 4, 6, 8]
    abundance_rows = []
    active_time = list(all_results['Active'][target_formate]['time'])
    for day in sampling_days:
        hour = day * 24
        idx = active_time.index(hour)
        active_biomass = all_results['Active'][target_formate]['X'][idx]
        diffusion_biomass = all_results['Diffusion'][target_formate]['X'][idx]
        if day == 0:
            active_abundance = 50.0
            diffusion_abundance = 50.0
        else:
            active_delta = max(0.0, active_biomass - x0)
            diffusion_delta = max(0.0, diffusion_biomass - x0)
            total_delta = active_delta + diffusion_delta
            if total_delta > 0:
                active_abundance = 100.0 * active_delta / total_delta
                diffusion_abundance = 100.0 * diffusion_delta / total_delta
            else:
                active_abundance = 50.0
                diffusion_abundance = 50.0
        abundance_rows.append({
            'Time_d': day,
            'DSM123_WT_Active_percent': round(active_abundance, 4),
            'Delta_forT_Diffusion_percent': round(diffusion_abundance, 4),
        })

    abundance_data = pd.DataFrame(abundance_rows)
    abundance_data.to_csv(output_dir / 'mm_dfba_20mM_abundance_bar_data.csv', index=False)

    fig_bar, ax_bar = plt.subplots(figsize=(7, 6.5))
    labels = [f'day {int(day)}' for day in abundance_data['Time_d']]
    x_pos = np.arange(len(labels))
    width = 0.35
    color_active = '#b01a52'
    color_diffusion = '#135263'

    ax_bar.bar(
        x_pos - width / 2,
        abundance_data['Delta_forT_Diffusion_percent'],
        width,
        label=r'$\Delta forT$',
        color=color_diffusion,
        edgecolor='black',
        linewidth=1,
    )
    ax_bar.bar(
        x_pos + width / 2,
        abundance_data['DSM123_WT_Active_percent'],
        width,
        label='DSM123 (WT)',
        color=color_active,
        edgecolor='black',
        linewidth=1,
    )
    for spine in ['left', 'bottom']:
        ax_bar.spines[spine].set_linewidth(2)
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.set_ylabel('Relative abundance (%)', fontsize=15, fontweight='bold')
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(labels, fontsize=13)
    ax_bar.set_ylim(0, 110)
    ax_bar.tick_params(width=2, length=6, labelsize=13)
    ax_bar.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False, fontsize=13)
    plt.tight_layout()
    abundance_figure = output_dir / 'mm_dfba_20mM_abundance_evolution_bars.svg'
    def simulate_fort_variant(km_value, vmax_value, variant_label, parameter_type, parameter_multiplier):
        x, formate, co2 = x0, float(target_formate), 0.0
        variant_history = {'time': [], 'X': [], 'For': [], 'CO2': []}
        variant_records = []
        for t in np.arange(0, t_max + dt, dt):
            if formate > 1e-4:
                v_uptake = vmax_value * (formate / (km_value + formate))
            else:
                v_uptake = 0.0
            concentration = _concentration_map(formate, co2)
            with base_model as m:
                apply_mm_constraints(m, concentration, t, 'ForT sensitivity', target_formate)
                m.reactions.get_by_id('EX_for_e').lower_bound = -v_uptake
                if 'EX_photon_purple_e' in m.reactions:
                    m.reactions.get_by_id('EX_photon_purple_e').lower_bound = -min(max_specific_light, total_light / max(x, 1e-12))
                sol_value = m.slim_optimize()
                if sol_value and sol_value > 1e-9:
                    full_sol = m.optimize()
                    mu = float(full_sol.objective_value)
                    f_for = float(full_sol.fluxes.get('EX_for_e', 0.0))
                    f_co2 = float(full_sol.fluxes.get('EX_co2_e', 0.0))
                else:
                    mu, f_for, f_co2 = 0.0, 0.0, 0.0
            variant_history['time'].append(t)
            variant_history['X'].append(x)
            variant_history['For'].append(formate)
            variant_history['CO2'].append(co2)
            variant_records.append({
                'parameter_type': parameter_type,
                'parameter_multiplier': parameter_multiplier,
                'variant_label': variant_label,
                'time_h': t,
                'time_d': t / 24.0,
                'fort_km_mM': km_value,
                'fort_effective_vmax_mmol_gDW_h': vmax_value,
                'biomass_gDW_L': x,
                'formate_mM': formate,
                'formate_uptake_capacity_mmol_gDW_h': v_uptake,
                'mu_h_inv': mu,
            })
            x += mu * x * dt
            formate += f_for * x * dt
            co2 += f_co2 * x * dt
            if formate < 0:
                formate = 0.0
        return variant_history, variant_records

    fort_sensitivity_specs = {
        'Km': [
            {'multiplier': 0.1, 'km': km_fort * 0.1, 'vmax': vmax_transport, 'label': 'Km 0.1x'},
            {'multiplier': 1.0, 'km': km_fort, 'vmax': vmax_transport, 'label': 'Km 1x'},
            {'multiplier': 10.0, 'km': km_fort * 10.0, 'vmax': vmax_transport, 'label': 'Km 10x'},
            {'multiplier': 100.0, 'km': km_fort * 100.0, 'vmax': vmax_transport, 'label': 'Km 100x'},
            {'multiplier': 200.0, 'km': km_fort * 200.0, 'vmax': vmax_transport, 'label': 'Km 200x'},
        ],
        'Vmax': [
            {'multiplier': 0.01, 'km': km_fort, 'vmax': vmax_transport * 0.01, 'label': 'Vmax 0.01x'},
            {'multiplier': 0.05, 'km': km_fort, 'vmax': vmax_transport * 0.05, 'label': 'Vmax 0.05x'},
            {'multiplier': 0.1, 'km': km_fort, 'vmax': vmax_transport * 0.1, 'label': 'Vmax 0.1x'},
            {'multiplier': 0.2, 'km': km_fort, 'vmax': vmax_transport * 0.2, 'label': 'Vmax 0.2x'},
            {'multiplier': 0.5, 'km': km_fort, 'vmax': vmax_transport * 0.5, 'label': 'Vmax 0.5x'},
            {'multiplier': 1.0, 'km': km_fort, 'vmax': vmax_transport, 'label': 'Vmax 1x'},
        ],
    }
    sensitivity_results = {}
    sensitivity_records = []
    for parameter_type, specs in fort_sensitivity_specs.items():
        sensitivity_results[parameter_type] = []
        for spec in specs:
            variant_history, variant_records = simulate_fort_variant(
                spec['km'], spec['vmax'], spec['label'], parameter_type, spec['multiplier']
            )
            sensitivity_results[parameter_type].append((spec, variant_history))
            sensitivity_records.extend(variant_records)

    sensitivity_df = pd.DataFrame(sensitivity_records)
    sensitivity_df.to_csv(output_dir / 'mm_dfba_20mM_fort_parameter_sensitivity_data.csv', index=False)
    fig_sens, axes_sens = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True)
    palette = ['#3b7ea1', '#5aa469', '#c2185b', '#e69f00', '#7b3294', '#444444']
    panel_titles = {
        'Km': 'ForT Km sensitivity',
        'Vmax': 'ForT Vmax sensitivity',
    }
    for ax, (parameter_type, entries) in zip(axes_sens, sensitivity_results.items()):
        ax2 = ax.twinx()
        biomass_lines, biomass_labels = [], []
        formate_lines, formate_labels = [], []
        for color, (spec, history) in zip(palette, entries):
            time_days = np.array(history['time'], dtype=float) / 24.0
            biomass_line, = ax.plot(
                time_days,
                history['X'],
                color=color,
                linestyle='-',
                linewidth=2.6,
                label=f"{spec['label']} biomass",
            )
            formate_line, = ax2.plot(
                time_days,
                history['For'],
                color=color,
                linestyle='--',
                linewidth=2.0,
                alpha=0.9,
                label=f"{spec['label']} formate",
            )
            biomass_lines.append(biomass_line)
            biomass_labels.append(spec['label'])
            formate_lines.append(formate_line)
            formate_labels.append(f"{spec['label']} formate")
        ax.set_title(panel_titles[parameter_type], fontsize=14, fontweight='bold', pad=12)
        ax.set_xlabel('Time (d)', fontsize=12, fontweight='bold')
        ax.set_xlim(0, t_max / 24.0)
        ax.set_ylim(0, 0.55)
        ax2.set_ylim(-0.5, target_formate * 1.08)
        ax.grid(True, linestyle=':', alpha=0.42)
        if ax is axes_sens[0]:
            ax.set_ylabel(r'Biomass (gDW L$^{-1}$)', fontsize=12, fontweight='bold')
        if ax is axes_sens[-1]:
            ax2.set_ylabel('Formate remaining (mM)', fontsize=12, fontweight='bold')
        else:
            ax2.set_yticklabels([])
        ax.legend(biomass_lines + formate_lines, biomass_labels + formate_labels, loc='upper left', fontsize=7, frameon=True)

    fig_sens.suptitle('20 mM dFBA response to ForT Km and Vmax', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    sensitivity_figure = output_dir / 'mm_dfba_20mM_fort_parameter_sensitivity.svg'
    fig_sens.savefig(sensitivity_figure, format='svg', dpi=300, bbox_inches='tight')
    plt.close(fig_sens)

    fva_vmax_specs = fort_sensitivity_specs['Vmax']
    fva_growth_fraction = 0.90
    rubisco_representative = 'RBPCcx'
    rubisco_disabled_reactions = {'RBCh', 'RBPC'}
    fva_heatmap_excluded_reactions = {'ACONTa', 'ACONTb'}
    fva_reaction_rows = ['EX_for_e', 'FDH', rubisco_representative] + [
        rid for rid in constrained_reactions
        if rid not in {'FDH', rubisco_representative} and rid not in rubisco_disabled_reactions and rid not in fva_heatmap_excluded_reactions
    ]
    fva_panel_a_metrics = {
        'Biomass': 'BIOMASS__1',
        'Formate uptake': 'EX_for_e',
        'FDH': 'FDH',
        'RuBisCO carboxylation': rubisco_representative,
    }
    fva_all_reactions = list(dict.fromkeys(list(fva_panel_a_metrics.values()) + fva_reaction_rows))
    fva_records = []
    for spec in fva_vmax_specs:
        formate_for_fva = float(target_formate)
        v_uptake = spec['vmax'] * (formate_for_fva / (km_fort + formate_for_fva))
        concentration = _concentration_map(formate_for_fva, 0.0)
        with base_model as m:
            apply_mm_constraints(m, concentration, 0.0, 'ForT Vmax loopless FVA', target_formate)
            fva_disabled_reactions = set(rubisco_disabled_reactions)
            for duplicate_rid in ['PGK_1', 'GAPDi_nadp', 'ACONTa', 'ACONTb']:
                if duplicate_rid in m.reactions:
                    fva_disabled_reactions.add(duplicate_rid)
            for disabled_rid in fva_disabled_reactions:
                if disabled_rid in m.reactions:
                    m.reactions.get_by_id(disabled_rid).bounds = (0.0, 0.0)
            if 'EX_co2_e' in m.reactions:
                m.reactions.get_by_id('EX_co2_e').lower_bound = 0.0
            m.reactions.get_by_id('EX_for_e').lower_bound = -v_uptake
            if 'EX_photon_purple_e' in m.reactions:
                m.reactions.get_by_id('EX_photon_purple_e').bounds = (-max_specific_light, 0.0)
            full_sol = m.optimize()
            if full_sol.status == 'optimal':
                optimum_fluxes = full_sol.fluxes
                optimum_biomass = float(full_sol.objective_value)
            else:
                optimum_fluxes = pd.Series(dtype=float)
                optimum_biomass = 0.0
            reaction_objs = [m.reactions.get_by_id(rid) for rid in fva_all_reactions if rid in m.reactions]
            fva_result = cobra.flux_analysis.flux_variability_analysis(
                m,
                reaction_list=reaction_objs,
                fraction_of_optimum=fva_growth_fraction,
                loopless=True,
                processes=1,
            )
        for rid in fva_all_reactions:
            if rid not in fva_result.index:
                continue
            flux_min = float(fva_result.loc[rid, 'minimum'])
            flux_max = float(fva_result.loc[rid, 'maximum'])
            opt_flux = optimum_biomass if rid == 'BIOMASS__1' else float(optimum_fluxes.get(rid, 0.0))
            if rid == 'EX_for_e':
                feasible_min = max(0.0, -flux_max)
                feasible_max = max(0.0, -flux_min)
                optimal_value = max(0.0, -opt_flux)
                width = feasible_max - feasible_min
                quantity = 'positive uptake'
            else:
                feasible_min = flux_min
                feasible_max = flux_max
                optimal_value = opt_flux
                width = flux_max - flux_min
                quantity = 'flux'
            fva_records.append({
                'fva_growth_fraction_of_optimum': fva_growth_fraction,
                'fva_loopless': True,
                'rubisco_representative': rubisco_representative,
                'disabled_rubisco_like_reactions': ';'.join(sorted(rubisco_disabled_reactions)),
                'disabled_duplicate_reactions': ';'.join([rid for rid in ['PGK_1', 'GAPDi_nadp', 'ACONTa', 'ACONTb'] if rid in base_model.reactions]),
                'heatmap_excluded_reactions': ';'.join(sorted(fva_heatmap_excluded_reactions)),
                'vmax_multiplier': spec['multiplier'],
                'vmax_label': spec['label'],
                'fort_effective_vmax_mmol_gDW_h': spec['vmax'],
                'fort_uptake_capacity_at_20mM_mmol_gDW_h': v_uptake,
                'reaction_id': rid,
                'quantity': quantity,
                'fva_minimum_flux': flux_min,
                'fva_maximum_flux': flux_max,
                'feasible_minimum': feasible_min,
                'feasible_maximum': feasible_max,
                'growth_optimal_value': optimal_value,
                'fva_width': width,
            })

    fva_df = pd.DataFrame(fva_records)
    fva_raw_export_columns = [
        'reaction_id',
        'vmax_multiplier',
        'vmax_label',
        'quantity',
        'feasible_minimum',
        'feasible_maximum',
        'growth_optimal_value',
        'fva_width',
    ]
    fva_df[fva_raw_export_columns].to_csv(
        output_dir / 'mm_fva_transport_gated_capacity_raw_values.csv',
        index=False,
        float_format='%.6g',
    )

    panel_a_rows = []
    for metric_name, rid in fva_panel_a_metrics.items():
        subset = fva_df[fva_df['reaction_id'] == rid].copy()
        if metric_name == 'RuBisCO carboxylation':
            scale = max(subset['feasible_maximum'].abs().max(), subset['growth_optimal_value'].abs().max())
        else:
            scale = max(subset['feasible_maximum'].abs().max(), subset['growth_optimal_value'].abs().max())
        if not np.isfinite(scale) or scale <= 0:
            scale = 1.0
        subset['panel_metric'] = metric_name
        subset['normalized_feasible_minimum'] = subset['feasible_minimum'] / scale
        subset['normalized_feasible_maximum'] = subset['feasible_maximum'] / scale
        subset['normalized_growth_optimal_value'] = subset['growth_optimal_value'] / scale
        subset['normalization_scale'] = scale
        panel_a_rows.append(subset)
    panel_a_df = pd.concat(panel_a_rows, ignore_index=True)
    panel_a_export_columns = [
        'panel_metric',
        'reaction_id',
        'vmax_multiplier',
        'vmax_label',
        'feasible_minimum',
        'feasible_maximum',
        'growth_optimal_value',
        'fva_width',
        'normalized_feasible_minimum',
        'normalized_feasible_maximum',
        'normalized_growth_optimal_value',
    ]
    panel_a_df[panel_a_export_columns].to_csv(
        output_dir / 'mm_fva_transport_gated_capacity_panel_a_values.csv',
        index=False,
        float_format='%.6g',
    )

    width_matrix = fva_df[fva_df['reaction_id'].isin(fva_reaction_rows)].pivot_table(
        index='reaction_id', columns='vmax_multiplier', values='fva_width', aggfunc='first'
    ).reindex(fva_reaction_rows)
    width_matrix = width_matrix[[spec['multiplier'] for spec in fva_vmax_specs]]

    normalized_width_matrix = width_matrix.div(width_matrix.max(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    normalized_width_matrix.to_csv(output_dir / 'mm_fva_transport_gated_capacity_normalized_width_values.csv', float_format='%.6g')

    demand_matrix = fva_df[fva_df['reaction_id'].isin(fva_reaction_rows)].pivot_table(
        index='reaction_id', columns='vmax_multiplier', values='growth_optimal_value', aggfunc='first'
    ).reindex(fva_reaction_rows)
    demand_matrix = demand_matrix[[spec['multiplier'] for spec in fva_vmax_specs]].abs()
    normalized_demand_matrix = demand_matrix.div(demand_matrix.max(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    normalized_demand_matrix.to_csv(output_dir / 'mm_fva_transport_gated_capacity_normalized_growth_demand_values.csv', float_format='%.6g')

    fig_height = max(7.2, 0.32 * len(fva_reaction_rows) + 2.6)
    fig_fva, (ax_fva_a, ax_fva_b) = plt.subplots(1, 2, figsize=(14.8, fig_height), gridspec_kw={'width_ratios': [1.08, 1.12]})
    metric_colors = {
        'Biomass': '#1f77b4',
        'Formate uptake': '#c2185b',
        'FDH': '#2ca02c',
        'RuBisCO carboxylation': '#ff7f0e',
    }
    metric_styles = {
        'Biomass': '-.',
        'Formate uptake': '-',
        'FDH': '--',
        'RuBisCO carboxylation': ':',
    }
    x_values = np.array([spec['multiplier'] for spec in fva_vmax_specs], dtype=float)
    metric_plot_order = [name for name in fva_panel_a_metrics if name != 'Biomass'] + ['Biomass']
    metric_markers = {
        'Biomass': 'D',
        'Formate uptake': 'o',
        'FDH': 's',
        'RuBisCO carboxylation': '^',
    }
    for metric_name in metric_plot_order:
        subset = panel_a_df[panel_a_df['panel_metric'] == metric_name].sort_values('vmax_multiplier')
        y_min = subset['normalized_feasible_minimum'].to_numpy(dtype=float)
        y_max = subset['normalized_feasible_maximum'].to_numpy(dtype=float)
        y_opt = subset['normalized_growth_optimal_value'].to_numpy(dtype=float)
        color = metric_colors[metric_name]
        zorder = 8 if metric_name == 'Biomass' else 3
        alpha = 0.22 if metric_name == 'Biomass' else 0.10
        linewidth = 3.3 if metric_name == 'Biomass' else 2.4
        ax_fva_a.fill_between(x_values, y_min, y_max, color=color, alpha=alpha, linewidth=0, zorder=zorder - 1)
        ax_fva_a.plot(
            x_values,
            y_opt,
            color=color,
            linestyle=metric_styles[metric_name],
            linewidth=linewidth,
            marker=metric_markers[metric_name],
            markersize=6.5,
            markeredgecolor='white',
            markeredgewidth=0.8,
            label=metric_name,
            zorder=zorder,
        )
    ax_fva_a.set_xscale('log')
    ax_fva_a.set_xticks(x_values)
    ax_fva_a.set_xticklabels([spec['label'].replace('Vmax ', '') for spec in fva_vmax_specs], rotation=30, ha='right')
    ax_fva_a.set_xlabel('ForT Vmax multiplier', fontsize=12, fontweight='bold')
    ax_fva_a.set_ylabel('Normalized loopless FVA feasible range / optimal flux', fontsize=12, fontweight='bold')
    ax_fva_a.set_title('A  Loopless FVA capacity under transport gating', fontsize=14, fontweight='bold', loc='left')
    ax_fva_a.set_ylim(-0.05, 1.08)
    ax_fva_a.grid(True, linestyle=':', alpha=0.40)
    ax_fva_a.legend(frameon=True, fontsize=9, loc='lower right')
    ax_fva_a.text(
        0.02, 0.02,
        f'Loopless FVA >= {int(fva_growth_fraction * 100)}% optimum; CO2 uptake blocked; duplicate PGK/GAPD/ACONT branches disabled; CS retained',
        transform=ax_fva_a.transAxes,
        fontsize=8.4,
        color='dimgray',
    )

    heatmap_values = normalized_width_matrix.to_numpy(dtype=float)
    im = ax_fva_b.imshow(heatmap_values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax_fva_b.set_title('B  Normalized loopless FVA width', fontsize=14, fontweight='bold', loc='left')
    ax_fva_b.set_xticks(np.arange(len(fva_vmax_specs)))
    ax_fva_b.set_xticklabels([spec['label'].replace('Vmax ', '') for spec in fva_vmax_specs], rotation=30, ha='right')
    ax_fva_b.set_yticks(np.arange(len(fva_reaction_rows)))
    ax_fva_b.set_yticklabels(fva_reaction_rows, fontsize=8)
    ax_fva_b.set_xlabel('ForT Vmax multiplier', fontsize=12, fontweight='bold')
    for i in range(heatmap_values.shape[0]):
        for j in range(heatmap_values.shape[1]):
            ax_fva_b.text(j, i, f'{heatmap_values[i, j]:.2f}', ha='center', va='center', fontsize=6, color='black')
    cbar = fig_fva.colorbar(im, ax=ax_fva_b, fraction=0.046, pad=0.04)
    cbar.set_label('Normalized loopless FVA width\n(low = constrained, high = flexible)', fontsize=10)
    fig_fva.suptitle('FVA reveals transport-gated carbon flux capacity', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    fva_figure = output_dir / 'mm_fva_transport_gated_carbon_flux_capacity_loopless_filtered.svg'
    fig_fva.savefig(fva_figure, format='svg', dpi=300, bbox_inches='tight')
    plt.close(fig_fva)

    print(f'Michaelis-Menten enzyme-constrained dFBA outputs saved to: {output_dir.resolve()}')
    print(f'Michaelis-Menten bounds were applied to {len(kinetics_df)} CBB/TCA/FDH reactions at every dFBA step.')
    return kinetics_df, substrate_df, bounds_df, summary_df


from urllib.parse import quote_plus


def _uniprot_query_link(row):
    """Build a reproducible UniProt search link without inventing an accession."""
    query_terms = []
    for field in ['uniprot_accession_or_query', 'rpal_homolog_locus_tags', 'enzyme_name', 'ec_number']:
        value = row.get(field, '')
        if pd.notna(value) and str(value).strip():
            query_terms.append(str(value).replace('query:', '').strip())
    query_terms.append('Rhodopseudomonas palustris')
    return 'https://www.uniprot.org/uniprotkb?query=' + quote_plus(' '.join(query_terms))


def build_final_mm_parameter_summary(kinetics_df, substrate_df):
    """Create a compact table that only exposes experimentally traceable kinetic constants."""
    rows = []
    substrate_groups = {rid: group.copy() for rid, group in substrate_df.groupby('reaction_id')}
    for _, row in kinetics_df.iterrows():
        rid = row['reaction_id']
        is_used = bool(row.get('use_mm_constraint', False))
        subs = substrate_groups.get(rid, pd.DataFrame())
        if is_used:
            km_text = '; '.join(
                f"{s.metabolite_id}: {float(s.km_mM):.6g} mM"
                for s in subs.itertuples(index=False)
            )
            substrate_text = '; '.join(
                f"{s.metabolite_id}: {float(s.default_concentration_mM):.6g} mM"
                for s in subs.itertuples(index=False)
            )
            kcat_value = row['kcat_s_inv']
            vmax_value = row['vmax_mmol_gdw_h']
            formula = 'Vmax * min(S_i / (Km_i + S_i))'
        else:
            km_text = 'NA: real kinetic record not verified'
            substrate_text = 'not used for MM bound'
            kcat_value = np.nan
            vmax_value = np.nan
            formula = 'not applied'
        rows.append({
            'reaction_id': rid,
            'pathway': row['pathway'],
            'enzyme_name': row['enzyme_name'],
            'ec_number': row['ec_number'],
            'dsm123_locus_tags': row['dsm_locus_tags'],
            'pgidnb_rnaseq_locus_tags': row['pgidnb_locus_tags'],
            'kinetic_record_status': row.get('kinetic_record_status', 'not verified'),
            'kinetic_reference_accession': row['kinetic_reference_accession'] if is_used else 'NA',
            'kinetic_reference_organism_strain': row['kinetic_reference_organism_strain'] if is_used else 'NA',
            'kinetic_reference_note': row['kinetic_reference_note'],
            'kcat_s_inv': kcat_value,
            'km_mM_by_substrate': km_text,
            'substrate_pool_mM_used_for_MM': substrate_text,
            'rnaseq_6h_median_expression': row['rnaseq_6h_median_expression'],
            'rnaseq_allocation_fraction': row['rnaseq_allocation_fraction'],
            'enzyme_umol_per_gDW': row['enzyme_umol_per_gdw'],
            'vmax_mmol_gDW_h': vmax_value,
            'mm_bound_formula': formula,
        })
    return pd.DataFrame(rows)

def cleanup_mm_output_dir(output_dir):
    """Preserve curated downstream outputs in the GitHub-facing analysis folder."""
    return []


mm_kinetics_df, mm_substrate_df, mm_bounds_df, mm_dfba_summary = run_mm_enzyme_constrained_dfba(MODEL_JSON, MM_DFBA_OUTPUT_DIR)
mm_parameter_summary = build_final_mm_parameter_summary(mm_kinetics_df, mm_substrate_df)
final_parameter_path = MM_DFBA_OUTPUT_DIR / 'reaction_parameters.csv'
mm_parameter_summary.to_csv(final_parameter_path, index=False, encoding='utf-8-sig')
locked_files = cleanup_mm_output_dir(MM_DFBA_OUTPUT_DIR)

print(f'Final parameter table: {final_parameter_path.resolve()}')
print(f'Final curve figure: {(MM_DFBA_OUTPUT_DIR / "mm_enzyme_constrained_dfba_20mM_active_diffusion.svg").resolve()}')
print(f'Final abundance figure: {(MM_DFBA_OUTPUT_DIR / "mm_dfba_20mM_abundance_evolution_bars.svg").resolve()}')
print(f'ForT sensitivity figure: {(MM_DFBA_OUTPUT_DIR / "mm_dfba_20mM_fort_parameter_sensitivity.svg").resolve()}')
print(f'FVA transport-gated figure: {(MM_DFBA_OUTPUT_DIR / "mm_fva_transport_gated_carbon_flux_capacity_loopless_filtered.svg").resolve()}')
if locked_files:
    print('Files kept because they appear to be open/locked:', locked_files)

display(mm_parameter_summary[[
    'reaction_id', 'pathway', 'enzyme_name', 'ec_number', 'kinetic_record_status',
    'kinetic_reference_accession', 'kcat_s_inv', 'km_mM_by_substrate',
    'rnaseq_allocation_fraction', 'enzyme_umol_per_gDW', 'vmax_mmol_gDW_h'
]])
display(mm_dfba_summary)


## Module capacity scaling under enzyme-constrained dFBA

This downstream analysis independently scales ForT transport, FDH, CBB, TCA, cyclic electron flow, and photon-uptake capacities under four ForT backgrounds. The simulation code generates the summary table, and the plotting code renders the response heatmaps used in the supplementary analysis. Set `MODULE_FORCE_RECOMPUTE = True` to rerun all 144 dFBA conditions; otherwise, the existing summary is used to regenerate the figures.


In [ ]:
# MODULE_CAPACITY_SCALING_INTEGRATED_V1
# Integrated from plot_mm_dfba_module_capacity_scaling.py and
# plot_mm_dfba_module_capacity_scaling_heatmap_from_summary.py.

MODULE_SCALES = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
MODULE_FORT_BACKGROUNDS = [1.0]
MODULE_INITIAL_FORMATE_MM = 20.0
MODULE_VMAX_TRANSPORT = 1.786555464
MODULE_KM_FORT = 0.01814
MODULE_T_MAX_H = 200
MODULE_DT_H = 2.0
MODULE_X0_GDW_L = 0.02
MODULE_TOTAL_LIGHT = 50.0
MODULE_MAX_SPECIFIC_LIGHT = 60.0
MODULE_BASELINE_CEF_CAPACITY = 60.0
MODULE_BLOCK_EXTERNAL_CO2_UPTAKE = True
MODULE_FORCE_RECOMPUTE = False

MODULE_ORDER = [
    "ForT transport",
    "FDH",
    "CBB",
    "TCA",
    "Cyclic electron flow",
    "Photon uptake",
]
# Keep all modules in the comparative heatmap.
# ForT transport should be interpreted as effective capacity = ForT background x module scale.
MODULE_PLOT_ORDER = MODULE_ORDER.copy()
MODULE_DATA_CSV = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_capacity_scaling_data.csv"
MODULE_FIG_SVG = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_capacity_scaling_response_heatmap.svg"
MODULE_FIG_PNG = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_capacity_scaling_response_heatmap.png"
MODULE_FIG_CS_SVG = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_capacity_scaling_response_heatmap_CS_standard.svg"
MODULE_FIG_CS_PNG = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_capacity_scaling_response_heatmap_CS_standard.png"
MODULE_RNASEQ_XLSX = MM_DFBA_OUTPUT_DIR / "Supplementary File S1. DSM 123 RNA-seq DEG.xlsx"


def _module_scale_kinetics(module_name, scale, baseline_kcat):
    """Scale kcat values for one kinetic module and return affected reactions."""
    scaled = []
    for rid, kinetic in KINETIC_ASSUMPTIONS.items():
        kinetic["kcat_s_inv"] = baseline_kcat[rid]
        if not kinetic.get("use_mm_constraint", False):
            continue
        if module_name == "FDH" and rid == "FDH":
            kinetic["kcat_s_inv"] = baseline_kcat[rid] * scale
            scaled.append(rid)
        elif module_name == "CBB" and kinetic.get("pathway") == "CBB":
            kinetic["kcat_s_inv"] = baseline_kcat[rid] * scale
            scaled.append(rid)
        elif module_name == "TCA" and kinetic.get("pathway") == "TCA":
            kinetic["kcat_s_inv"] = baseline_kcat[rid] * scale
            scaled.append(rid)
    return scaled


def _module_restore_kinetics(baseline_kcat):
    for rid, value in baseline_kcat.items():
        KINETIC_ASSUMPTIONS[rid]["kcat_s_inv"] = value


def _module_prepare_base_model():
    base_model = cobra.io.load_json_model(str(MODEL_JSON))
    base_model = apply_base_conditions(base_model)
    base_model.objective = "BIOMASS__1"
    if MODULE_BLOCK_EXTERNAL_CO2_UPTAKE and "EX_co2_e" in base_model.reactions:
        base_model.reactions.get_by_id("EX_co2_e").lower_bound = 0.0
    apply_rnaseq_enzyme_scaling(
        base_model,
        MODULE_RNASEQ_XLSX,
        enzyme_pool_fraction_of_biomass_protein=1.0,
    )
    return base_model


def _module_apply_non_enzyme_capacity(m, module_name, scale, max_specific_light):
    """Apply the exact cyclic-electron-flow and photon bounds used for the figure."""
    if module_name == "Cyclic electron flow" and "PURPLE_RC" in m.reactions:
        reaction = m.reactions.get_by_id("PURPLE_RC")
        reaction.upper_bound = min(reaction.upper_bound, MODULE_BASELINE_CEF_CAPACITY * scale)
    if "EX_photon_purple_e" in m.reactions:
        photon_bound = max_specific_light
        if module_name == "Photon uptake":
            photon_bound *= scale
        m.reactions.get_by_id("EX_photon_purple_e").lower_bound = -photon_bound


def _module_run_single_dfba(base_model, module_name, module_scale, fort_background, baseline_kcat):
    scaled_reactions = _module_scale_kinetics(module_name, module_scale, baseline_kcat)
    total_light = MODULE_TOTAL_LIGHT * (module_scale if module_name == "Photon uptake" else 1.0)
    max_specific_light = MODULE_MAX_SPECIFIC_LIGHT * (module_scale if module_name == "Photon uptake" else 1.0)
    fort_vmax = MODULE_VMAX_TRANSPORT * fort_background
    if module_name == "ForT transport":
        fort_vmax *= module_scale

    biomass = MODULE_X0_GDW_L
    formate = MODULE_INITIAL_FORMATE_MM
    co2 = 0.0
    cumulative_biomass = 0.0
    cumulative_formate_consumed = 0.0
    cumulative_fdh = 0.0
    cumulative_rubisco_total = 0.0
    cumulative_rbpcx = 0.0
    cumulative_cs = 0.0
    cumulative_icdhyr = 0.0

    for time_h in np.arange(0, MODULE_T_MAX_H + MODULE_DT_H, MODULE_DT_H):
        if formate > 1e-4:
            uptake_capacity = fort_vmax * formate / (MODULE_KM_FORT + formate)
        else:
            uptake_capacity = 0.0

        concentration = _concentration_map(formate, co2)
        with base_model as model_step:
            apply_mm_constraints(
                model_step,
                concentration,
                float(time_h),
                f"{module_name} scaling",
                MODULE_INITIAL_FORMATE_MM,
            )
            if MODULE_BLOCK_EXTERNAL_CO2_UPTAKE and "EX_co2_e" in model_step.reactions:
                model_step.reactions.get_by_id("EX_co2_e").lower_bound = 0.0
            model_step.reactions.get_by_id("EX_for_e").lower_bound = -uptake_capacity
            if "EX_photon_purple_e" in model_step.reactions:
                light_bound = min(max_specific_light, total_light / max(biomass, 1e-12))
                model_step.reactions.get_by_id("EX_photon_purple_e").lower_bound = -light_bound
            _module_apply_non_enzyme_capacity(model_step, module_name, module_scale, max_specific_light)

            solution = model_step.optimize()
            if solution.status == "optimal" and solution.objective_value and solution.objective_value > 1e-9:
                growth_rate = float(solution.objective_value)
                formate_flux = float(solution.fluxes.get("EX_for_e", 0.0))
                co2_flux = float(solution.fluxes.get("EX_co2_e", 0.0))
                fdh_flux = float(solution.fluxes.get("FDH", 0.0))
                rbpc_flux = float(solution.fluxes.get("RBPC", 0.0))
                rbpcx_flux = float(solution.fluxes.get("RBPCcx", 0.0))
                rubisco_total_flux = max(rbpc_flux, 0.0) + max(rbpcx_flux, 0.0)
                cs_flux = float(solution.fluxes.get("CS", 0.0))
                icdhyr_flux = float(solution.fluxes.get("ICDHyr", 0.0))
            else:
                growth_rate = formate_flux = co2_flux = fdh_flux = 0.0
                rubisco_total_flux = rbpcx_flux = cs_flux = icdhyr_flux = 0.0

        biomass_increment = growth_rate * biomass * MODULE_DT_H
        cumulative_biomass += biomass_increment
        cumulative_formate_consumed += max(-formate_flux, 0.0) * biomass * MODULE_DT_H
        cumulative_fdh += max(fdh_flux, 0.0) * biomass * MODULE_DT_H
        cumulative_rubisco_total += max(rubisco_total_flux, 0.0) * biomass * MODULE_DT_H
        cumulative_rbpcx += max(rbpcx_flux, 0.0) * biomass * MODULE_DT_H
        cumulative_cs += max(cs_flux, 0.0) * biomass * MODULE_DT_H
        cumulative_icdhyr += max(icdhyr_flux, 0.0) * biomass * MODULE_DT_H

        biomass += biomass_increment
        formate += formate_flux * biomass * MODULE_DT_H
        co2 += co2_flux * biomass * MODULE_DT_H
        if formate < 0:
            formate = 0.0

    return {
        "module": module_name,
        "module_scale": module_scale,
        "fort_background": fort_background,
        "final_biomass_gDW_L": biomass,
        "delta_biomass_gDW_L": biomass - MODULE_X0_GDW_L,
        "final_formate_mM": formate,
        "final_co2_mM": co2,
        "cumulative_biomass_production_gDW_L": cumulative_biomass,
        "cumulative_formate_consumed_mM_gDW_h_proxy": cumulative_formate_consumed,
        "cumulative_fdh_flux_mM_gDW_h_proxy": cumulative_fdh,
        "cumulative_rubisco_total_flux_mM_gDW_h_proxy": cumulative_rubisco_total,
        "cumulative_rbpcx_flux_mM_gDW_h_proxy": cumulative_rbpcx,
        "cumulative_cs_flux_mM_gDW_h_proxy": cumulative_cs,
        "cumulative_icdhyr_flux_mM_gDW_h_proxy": cumulative_icdhyr,
        "external_co2_uptake_blocked": MODULE_BLOCK_EXTERNAL_CO2_UPTAKE,
        "scaled_reactions": ";".join(scaled_reactions),
    }


def _module_normalize_to_1x(summary):
    baseline = summary[summary["module_scale"] == 1.0][
        [
            "module",
            "fort_background",
            "final_biomass_gDW_L",
            "cumulative_rubisco_total_flux_mM_gDW_h_proxy",
            "cumulative_rbpcx_flux_mM_gDW_h_proxy",
            "cumulative_cs_flux_mM_gDW_h_proxy",
            "cumulative_icdhyr_flux_mM_gDW_h_proxy",
        ]
    ].rename(
        columns={
            "final_biomass_gDW_L": "baseline_final_biomass_gDW_L",
            "cumulative_rubisco_total_flux_mM_gDW_h_proxy": "baseline_cumulative_rubisco_total",
            "cumulative_rbpcx_flux_mM_gDW_h_proxy": "baseline_cumulative_rbpcx",
            "cumulative_cs_flux_mM_gDW_h_proxy": "baseline_cumulative_cs",
            "cumulative_icdhyr_flux_mM_gDW_h_proxy": "baseline_cumulative_icdhyr",
        }
    )
    normalized = summary.merge(baseline, on=["module", "fort_background"], how="left")
    normalized["relative_final_biomass"] = normalized["final_biomass_gDW_L"] / normalized["baseline_final_biomass_gDW_L"].replace(0, np.nan)
    normalized["relative_cumulative_rubisco_total_flux"] = normalized["cumulative_rubisco_total_flux_mM_gDW_h_proxy"] / normalized["baseline_cumulative_rubisco_total"].replace(0, np.nan)
    normalized["relative_cumulative_rbpcx_flux"] = normalized["cumulative_rbpcx_flux_mM_gDW_h_proxy"] / normalized["baseline_cumulative_rbpcx"].replace(0, np.nan)
    normalized["relative_cumulative_cs_flux"] = normalized["cumulative_cs_flux_mM_gDW_h_proxy"] / normalized["baseline_cumulative_cs"].replace(0, np.nan)
    normalized["relative_cumulative_icdhyr_flux"] = normalized["cumulative_icdhyr_flux_mM_gDW_h_proxy"] / normalized["baseline_cumulative_icdhyr"].replace(0, np.nan)
    return normalized


def generate_module_capacity_summary(force=False):
    """Generate or load the 144-condition module-capacity data."""
    if MODULE_DATA_CSV.exists() and not force:
        return pd.read_csv(MODULE_DATA_CSV)

    base_model = _module_prepare_base_model()
    baseline_kcat = {rid: kinetic["kcat_s_inv"] for rid, kinetic in KINETIC_ASSUMPTIONS.items()}
    summary_rows = []
    try:
        for fort_background in MODULE_FORT_BACKGROUNDS:
            for module_name in MODULE_ORDER:
                for module_scale in MODULE_SCALES:
                    summary_rows.append(
                        _module_run_single_dfba(
                            base_model,
                            module_name,
                            module_scale,
                            fort_background,
                            baseline_kcat,
                        )
                    )
    finally:
        _module_restore_kinetics(baseline_kcat)

    summary = _module_normalize_to_1x(pd.DataFrame(summary_rows))
    summary.to_csv(MODULE_DATA_CSV, index=False, encoding="utf-8-sig")
    return summary


def _module_matrix_for_metric(summary, metric, fort_background):
    """Build module x scale matrix at one fixed ForT background."""
    matrix = []
    for module_name in MODULE_PLOT_ORDER:
        row = []
        for module_scale in MODULE_SCALES:
            subset = summary[
                (np.isclose(summary["fort_background"], fort_background))
                & (summary["module"] == module_name)
                & (np.isclose(summary["module_scale"], module_scale))
            ]
            row.append(float(subset[metric].iloc[0]) if len(subset) else np.nan)
        matrix.append(row)
    return np.asarray(matrix, dtype=float)


def _module_draw_metric_grid(axis, summary, metric, row_title, fort_background):
    """Draw one simple response heatmap: modules by module-capacity scale."""
    matrix = _module_matrix_for_metric(summary, metric, fort_background)
    image = axis.imshow(matrix, aspect="auto", cmap="RdBu_r", vmin=0, vmax=1)
    axis.set_title(row_title, fontsize=10, fontweight="bold")
    axis.set_xlabel("Module capacity scale", fontsize=9)
    axis.set_ylabel("Capacity-limited module", fontsize=9)
    axis.set_xticks(range(len(MODULE_SCALES)))
    axis.set_xticklabels([f"{scale:g}x" for scale in MODULE_SCALES], rotation=35, ha="right", fontsize=8)
    axis.set_yticks(range(len(MODULE_PLOT_ORDER)))
    axis.set_yticklabels(MODULE_PLOT_ORDER, fontsize=8)
    axis.tick_params(length=0)
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            if np.isfinite(value):
                text_color = "white" if value < 0.25 or value > 0.75 else "black"
                axis.text(column_index, row_index, f"{value:.2g}", ha="center", va="center", fontsize=7, color=text_color)
    return image


def plot_module_capacity_heatmap(summary, third_metric, third_label, svg_path, png_path):
    """Render the simplified three-panel module-capacity response heatmap."""
    numeric_columns = [
        "fort_background",
        "module_scale",
        "relative_final_biomass",
        "relative_cumulative_rubisco_total_flux",
        "relative_cumulative_icdhyr_flux",
        "relative_cumulative_cs_flux",
    ]
    summary = summary.copy()
    for column in numeric_columns:
        if column in summary.columns:
            summary[column] = pd.to_numeric(summary[column], errors="coerce")

    fort_background = MODULE_FORT_BACKGROUNDS[0]
    figure, axes = plt.subplots(1, 3, figsize=(13.8, 4.9), constrained_layout=True)
    _module_draw_metric_grid(axes[0], summary, "relative_final_biomass", "Final biomass", fort_background)
    _module_draw_metric_grid(axes[1], summary, "relative_cumulative_rubisco_total_flux", "Cumulative RuBisCO flux", fort_background)
    image = _module_draw_metric_grid(axes[2], summary, third_metric, third_label, fort_background)
    colorbar = figure.colorbar(image, ax=axes, fraction=0.025, pad=0.01)
    colorbar.set_label("Response relative to 1x module capacity", fontsize=8)
    colorbar.ax.tick_params(labelsize=8)
    figure.suptitle(
        f"Module capacity scaling response under enzyme-constrained dFBA "
        f"(ForT background {fort_background:g}x; t={MODULE_T_MAX_H} h; CO2 uptake blocked)",
        fontsize=12,
        fontweight="bold",
    )
    figure.savefig(svg_path, format="svg", bbox_inches="tight")
    figure.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close(figure)


module_capacity_summary = generate_module_capacity_summary(force=MODULE_FORCE_RECOMPUTE)
plot_module_capacity_heatmap(
    module_capacity_summary,
    "relative_cumulative_icdhyr_flux",
    "Cumulative ICDHyr flux",
    MODULE_FIG_SVG,
    MODULE_FIG_PNG,
)
plot_module_capacity_heatmap(
    module_capacity_summary,
    "relative_cumulative_cs_flux",
    "Cumulative CS flux",
    MODULE_FIG_CS_SVG,
    MODULE_FIG_CS_PNG,
)
display(module_capacity_summary.head(12))




In [ ]:
# Biomass sensitivity coefficient heatmap across adjacent capacity intervals.
# This post-processing step uses the same fixed ForT background and time horizon
# as the module-capacity response heatmap.

SENSITIVITY_MODULES = MODULE_ORDER.copy()
SENSITIVITY_FORT_BACKGROUND = MODULE_FORT_BACKGROUNDS[0]
SENSITIVITY_SCALES = MODULE_SCALES.copy()
SENSITIVITY_DATA_CSV = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_capacity_scaling_data.csv"
SENSITIVITY_FIG_SVG = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_biomass_sensitivity_coefficient_heatmap.svg"
SENSITIVITY_FIG_PNG = MM_DFBA_OUTPUT_DIR / "mm_dfba_module_biomass_sensitivity_coefficient_heatmap.png"


def calculate_biomass_sensitivity_coefficients(summary):
    """Calculate local log-elasticity of final biomass across adjacent capacity intervals."""
    summary = summary.copy()
    for column in ["module_scale", "fort_background", "final_biomass_gDW_L"]:
        summary[column] = pd.to_numeric(summary[column], errors="coerce")

    intervals = list(zip(SENSITIVITY_SCALES[:-1], SENSITIVITY_SCALES[1:]))
    columns = [f"{low:g}-{high:g}x" for low, high in intervals]
    sensitivity = pd.DataFrame(index=SENSITIVITY_MODULES, columns=columns, dtype=float)

    for module in SENSITIVITY_MODULES:
        subset = summary[
            (summary["module"] == module)
            & np.isclose(summary["fort_background"], SENSITIVITY_FORT_BACKGROUND)
        ]
        for low, high in intervals:
            y_low = subset.loc[np.isclose(subset["module_scale"], low), "final_biomass_gDW_L"]
            y_high = subset.loc[np.isclose(subset["module_scale"], high), "final_biomass_gDW_L"]
            if y_high.empty or y_low.empty or y_high.iloc[0] <= 0 or y_low.iloc[0] <= 0:
                sensitivity.loc[module, f"{low:g}-{high:g}x"] = np.nan
                continue
            value = (
                np.log(y_high.iloc[0]) - np.log(y_low.iloc[0])
            ) / (np.log(high) - np.log(low))
            sensitivity.loc[module, f"{low:g}-{high:g}x"] = max(0.0, value)

    return sensitivity


def plot_biomass_sensitivity_heatmap(sensitivity, svg_path, png_path):
    """Plot local biomass sensitivity coefficients across module-capacity intervals."""
    max_value = np.nanmax(sensitivity.to_numpy(dtype=float))
    colorbar_max = max(1.0, np.ceil(max_value * 10) / 10)

    fig, ax = plt.subplots(figsize=(8.4, 4.4), constrained_layout=True)
    heatmap = sns.heatmap(
        sensitivity,
        ax=ax,
        cmap="YlOrRd",
        vmin=0,
        vmax=colorbar_max,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Biomass sensitivity coefficient"},
    )
    ax.set_xlabel("Module capacity interval")
    ax.set_ylabel("Capacity-limited module")
    ax.set_title(
        f"Local biomass sensitivity to module capacity knockdown "
        f"(ForT background {SENSITIVITY_FORT_BACKGROUND:g}x; t={MODULE_T_MAX_H} h)",
        fontsize=11,
        fontweight="bold",
    )
    ax.tick_params(axis="x", labelrotation=35, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    heatmap.collections[0].colorbar.ax.tick_params(labelsize=8)
    heatmap.collections[0].colorbar.set_label(
        "S = ?ln(final biomass) / ?ln(module capacity)",
        fontsize=8,
    )
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()


sensitivity_summary = pd.read_csv(SENSITIVITY_DATA_CSV)
biomass_sensitivity = calculate_biomass_sensitivity_coefficients(sensitivity_summary)
display(biomass_sensitivity)
plot_biomass_sensitivity_heatmap(
    biomass_sensitivity,
    SENSITIVITY_FIG_SVG,
    SENSITIVITY_FIG_PNG,
)
print(f"Saved: {SENSITIVITY_FIG_SVG}")

